<a href="https://colab.research.google.com/github/eunyeongkimm/multimodal_user_needs_understanding/blob/main/results/09_audio_native_model_test/qwen3_omni_test_vllm_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===== 셀 1 : vLLM + PyTorch 호환 버전 설치 =====

!pip uninstall -y -q vllm vllm-omni torch torchvision torchaudio

# PyTorch / TorchAudio를 완전히 같은 버전 + CUDA 13.0으로 설치
!pip install -q --no-cache-dir \
    torch==2.11.0 \
    torchvision==0.26.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cu130

# vLLM 및 Qwen 오디오 의존성 설치
!pip install -q --no-cache-dir \
    vllm \
    qwen-omni-utils \
    soundfile \
    librosa

print("✅ 설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 136.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 133.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 114.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 140.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 121.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 304.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 123.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 341.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 143.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 122.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 140.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 120.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# CUDA 13 라이브러리 경로 등록
!echo "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib" \
    | sudo tee /etc/ld.so.conf.d/nvidia-cu13.conf

!sudo ldconfig

# 제대로 잡혔는지 확인
!ldconfig -p | grep libcudart

/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib
/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic

In [1]:
import torch
import torchaudio
import vllm

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("torchaudio:", torchaudio.__version__)
print("vllm:", vllm.__version__)

print("GPU:", torch.cuda.get_device_name(0))

torch: 2.13.0+cu130
CUDA: 13.0
torchaudio: 2.11.0+cu130
vllm: 0.27.1
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# ===== 셀 2 : 구글드라이브 마운트 =====
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, os
AUDIO_ROOT = '/content/drive/MyDrive/audio_seg'
PARQUET_DIR = '/content/drive/MyDrive/audio_seg_2'

manifest = pd.read_parquet(os.path.join(PARQUET_DIR, 'audio_seg_manifest.parquet'))
gold = pd.read_parquet(os.path.join(PARQUET_DIR, 'gold_actual_batch1_final.parquet'))
gold_map = dict(zip(gold.call_id, gold.label))
print("manifest:", manifest.shape, "| gold:", gold.shape)

Mounted at /content/drive
manifest: (1119, 11) | gold: (19847, 5)


In [3]:
import torch
print("free:", round(torch.cuda.mem_get_info()[0]/1e9,1), "GB / total:", round(torch.cuda.mem_get_info()[1]/1e9,1), "GB")

free: 42.0 GB / total: 42.4 GB


In [4]:
# ===== vLLM 로드: Colab 안전 버전 =====

import os
import sys

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# DEBUG 쓰지 않음
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# progress bar도 최대한 억제
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

MODEL_ID = "cyankiwi/Qwen3-Omni-30B-A3B-Thinking-AWQ-4bit"

# 원래 Colab 출력 보관
_original_stdout = sys.stdout
_original_stderr = sys.stderr

# 실제 파일 객체 → fileno() 지원
_vllm_log = open("/tmp/vllm_load.log", "w")

# ★ vLLM import 전에 변경
sys.stdout = _vllm_log
sys.stderr = _vllm_log

load_error = None

try:
    from vllm import LLM, SamplingParams

    llm = LLM(
        model=MODEL_ID,
        runner="generate",
        trust_remote_code=True,
        tensor_parallel_size=1,

        gpu_memory_utilization=0.90,
        max_model_len=4096,
        max_num_seqs=1,

        limit_mm_per_prompt={
            "audio": 5,
            "image": 1,
            "video": 0,
        },

        enforce_eager=True,
        generation_config="vllm",
    )

except Exception as e:
    load_error = e

finally:
    # Colab 화면 출력 복구
    sys.stdout = _original_stdout
    sys.stderr = _original_stderr

    # 일부 logger가 이 파일을 계속 참조할 수 있으므로
    # 여기서는 일부러 close하지 않음
    _vllm_log.flush()


if load_error is None:
    print("✅ vLLM loaded OK")

else:
    print("❌ vLLM load failed:")
    print(repr(load_error))

    # 전체 로그 절대 출력하지 않고 마지막 40줄만
    with open("/tmp/vllm_load.log", "r", errors="replace") as f:
        lines = f.readlines()

    print("\n===== 마지막 로그 40줄 =====")
    print("".join(lines[-40:]))

INFO 08-15 22:51:57 [api_utils.py:273] non-default args: {'runner': 'generate', 'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'audio': 5, 'image': 1, 'video': 0}, 'generation_config': 'vllm', 'model': 'cyankiwi/Qwen3-Omni-30B-A3B-Thinking-AWQ-4bit'}
WARNING 08-15 22:51:58 [arg_utils.py:1678] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 08-15 22:52:12 [model.py:645] Resolved architecture: Qwen3OmniMoeForConditionalGeneration
INFO 08-15 22:52:12 [model.py:1883] Using max model len 4096
INFO 08-15 22:52:18 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=4096.
WARNING 08-15 22:52:18 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-15 22:52:18 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-15 22:52:18 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 08-15 22:52:18 [vllm.py:1426] Cudagraph is disabled under eager mode
INFO 08-15 22:52:18 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant
INFO 08-15 22:52:36 [core.py:121] Initializing a V1 LLM en

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 08-15 22:54:07 [default_loader.py:430] Loading weights took 33.31 seconds
INFO 08-15 22:54:07 [int_wna16.py:409] Using MoEPrepareAndFinalizeNoDPEPModular
INFO 08-15 22:54:07 [int_wna16.py:410] Using MarlinExperts
INFO 08-15 22:54:10 [gpu_model_runner.py:5405] Model loading took 19.22 GiB memory and 90.392979 seconds
INFO 08-15 22:54:11 [gpu_model_runner.py:6465] Encoder cache will be initialized with a budget of 12544 tokens, and profiled with 1 image items of the maximum feature size.
INFO 08-15 22:55:57 [gpu_worker.py:563] Available KV cache memory: 14.11 GiB
INFO 08-15 22:55:57 [kv_cache_utils.py:2235] GPU KV cache size: 154,160 tokens
INFO 08-15 22:55:57 [kv_cache_utils.py:2236] Maximum concurrency for 4,096 tokens per request: 37.64x
INFO 08-15 22:55:57 [gpu_worker.py:789] Free memory on device (39.08/39.49 GiB) on startup. Desired GPU memory utilization is (0.9, 35.54 GiB). Actual usage is 19.91 GiB for consumed memory (weights + non-torch), 1.52 GiB for peak activation, and

In [5]:
# ===== 헬퍼 함수 복구 셀 =====

import os
import re

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]


def get_call_wavs(call_id, max_utterances=5):
    rows = (
        manifest[manifest.call_id == call_id]
        .sort_values("utt_idx")
        .head(max_utterances)
    )

    paths = []

    for _, r in rows.iterrows():
        p = os.path.join(
            AUDIO_ROOT,
            call_id,
            os.path.basename(r.wav_path)
        )

        if os.path.exists(p):
            paths.append(p)

    return paths


def build_prompt(n_audio):
    audio_tags = "".join(
        f"발화{i}: "
        "<|audio_start|><|audio_pad|><|audio_end|>\n"
        for i in range(1, n_audio + 1)
    )

    cats = ", ".join(CATEGORIES)

    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 통화 분류기입니다. "
        "고객의 초반 발화 음성을 듣고 의도를 분류합니다."
        "<|im_end|>\n"

        "<|im_start|>user\n"
        f"{audio_tags}\n"

        f"위 발화들을 종합해 고객 의도를 "
        f"다음 7개 중 하나로 분류하세요: {cats}\n"

        "최종 답변은 반드시 "
        "'정답: <카테고리>' 형식으로 작성하세요."
        "<|im_end|>\n"

        "<|im_start|>assistant\n"
    )


def parse_pred(resp):
    # "정답:" 뒤에 나오는 카테고리만 인정 (엄격)
    m = re.search(r"정답\s*[:：]\s*\n?\s*([가-힣]+)", resp)
    if m and m.group(1).strip() in CATEGORIES:
        return m.group(1).strip()
    # "정답:"이 아예 없고 응답이 안 잘렸으면(짧게 끝) 마지막 카테고리 허용
    # 단 "정답:"이 있는데 뒤가 빈 경우(잘림)는 None 처리
    if "정답" in resp:
        return None  # 정답 태그는 있는데 카테고리 못 뽑음 = 잘림/실패
    # 정답 태그 자체가 없으면 tail fallback
    tail = resp[-80:]
    for c in CATEGORIES:
        if c in tail:
            return c
    return None


print("✅ helper functions loaded")

✅ helper functions loaded


### 기본뼈대

In [7]:
# ===== 셀: 카테고리 정의 + 추론 허용=====
import time, librosa, re
from vllm import SamplingParams

CATEGORIES = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

# 카테고리 정의
CATEGORY_DEF = """
[카테고리 정의]
1. 환불요청: 결제한 금액을 돌려받는 것이 최종 목적인 경우로 보이면
   - "취소", "반품", "반송"이라는 단어가 나와도, 최종 목적이 금전 반환(현금/캐시)으로
     보이면 환불요청으로 분류할 것
2. 주문취소: 배송/수강 전, 환불 절차 없이 순수 주문 취소로 보이는 경우
3. 불만제기: 문의처럼 들리나 본질은 항의(약속 불이행, 응대 불만, 반복 통화에 대한
   불만 등)로 보이는 경우
4. 배송확인: 배송 상태·도착 문의로 보이는 경우
5. 교환반품: 불량·오배송으로 물건을 다른 물건으로 교체(금전 반환 아님)하려는
   것으로 보이는 경우
6. 구매진행: 결제 완료를 위한 도움을 요청하는 것으로 보이는 경우
7. 서비스이용: 로그인·기기·앱·시스템 이용 관련 문제로 보이는 경우
"""


# 추론 허용 — structured 안 씀, max_tokens 넉넉히
sampling = SamplingParams(temperature=0.0, max_tokens=3072)

def build_prompt(n_audio):
    audio_tags = "".join(
        f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|>\n" for i in range(1, n_audio+1)
    )
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 고객의 초반 발화 음성을 듣고 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{audio_tags}\n"
        "위 발화들을 듣고 고객 의도를 분류하세요.\n"
        "한국어로, 핵심 근거만 2문장 이내로 짧게 분석한 뒤 정답을 쓰세요.\n"
        "형식:\n"
        "분석: (2문장 이내)\n"
        "정답: (위 7개 중 하나)\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def parse_pred(resp):
    # "정답:" 우선, 없으면 응답 끝쪽 카테고리
    m = re.search(r"정답\s*[:：]\s*([가-힣]+)", resp)
    if m and m.group(1).strip() in CATEGORIES:
        return m.group(1).strip()
    tail = resp[-150:]
    for c in CATEGORIES:
        if c in tail:
            return c
    for c in CATEGORIES:
        if c in resp:
            return c
    return None

def predict_call(call_id):
    wavs = get_call_wavs(call_id, max_utterances=5)
    if not wavs:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "n_audio": 0}
    audios = [(librosa.load(w, sr=16000, mono=True)[0], 16000) for w in wavs]
    out = llm.generate(
        [{"prompt": build_prompt(len(audios)), "multi_modal_data": {"audio": audios}}],
        sampling_params=sampling, use_tqdm=False,
    )
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text,
            "finish": r.finish_reason, "n_tok": len(r.token_ids), "n_audio": len(audios)}

# ---- 5콜 테스트 (추론 붙는지 + 잘림 없는지 확인) ----
gold_map = dict(zip(gold.call_id, gold.label))
for cid in manifest.call_id.unique()[:5]:
    t0 = time.time()
    r = predict_call(cid)
    print(f"\n=== {cid}  gold={gold_map.get(cid)}  pred={r['pred']}  "
          f"({time.time()-t0:.1f}s, {r['n_tok']}tok, {r.get('finish')}) ===")
    print(r["raw"][:700])


=== J16_S000434  gold=서비스이용  pred=None  (182.8s, 2048tok, length) ===
<think>
Okay, let's tackle this problem. So, the user provided five speech segments in Korean, and I need to classify the customer's intent into one of the seven categories. Let me go through each utterance carefully.

First, the user says "여보세요. 아까는 뭐냐?" which translates to "Hello. What was that earlier?" Then the next part is "엠베스 쓰여가고 엘리아이 보이." Hmm, maybe "엠베스" is a typo or mispronunciation. Wait, maybe it's "엠베스" as in "Mebes" but that doesn't make sense. Alternatively, maybe it's "엠베스" as a brand or product name. Then "엘리아이 보이" might be "Eliai boi" but that's unclear. Maybe "엘리아이" is "Eliai" but perhaps it's a mispronunciation of "엘리트" or something else. Wait, maybe "엠베스" is "Mebes" but

=== J16_S000633  gold=서비스이용  pred=서비스이용  (54.6s, 605tok, stop) ===
<think>
Okay, let's tackle this problem. So, the user provided five speech segments in Korean, and I need to classify the customer's intent into one of the seve

### think-off

In [9]:
# ===== think-OFF: 정의 + 즉답 (빠름) =====
import time, librosa, re, pandas as pd
from vllm import SamplingParams

sampling_off = SamplingParams(temperature=0.0, max_tokens=512)

def build_prompt_off(n_audio):
    audio_tags = "".join(f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|>\n" for i in range(1, n_audio+1))
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 고객의 초반 발화 음성을 듣고 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{audio_tags}\n"
        "위 발화들을 듣고 고객 의도를 분류하세요.\n형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def predict_off(call_id):
    wavs = get_call_wavs(call_id, max_utterances=5)
    if not wavs:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(w, sr=16000, mono=True)[0], 16000) for w in wavs]
    out = llm.generate([{"prompt": build_prompt_off(len(audios)), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_off, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text, "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# 5콜 확인
for cid in manifest.call_id.unique()[:5]:
    r = predict_off(cid)
    print(f"{cid} gold={gold_map.get(cid)} pred={r['pred']} finish={r['finish']} tok={r['n_tok']}")
    print(r["raw"][:150], "\n")

J16_S000434 gold=서비스이용 pred=서비스이용 finish=stop tok=568
<think>
Okay, let's tackle this problem. So, the user provided five different utterances in Korean, and I need to classify the customer's intent into  

J16_S000633 gold=서비스이용 pred=서비스이용 finish=stop tok=88
분석: 고객은 과거에 기기 변경을 여러 번 했고, 이번에 다시 기기 변경을 시도했으나 로그인 문제가 발생해 진행이 어려워졌다고 설명하고 있습니다. 이는 로그인 및 기기 변경 과정에서 발생한 시스템 문제로 보이며, 서비스 이용 관련 문의로 판단됩니다.
정답: 서비스이용 

J16_S000687 gold=서비스이용 pred=서비스이용 finish=stop tok=78
분석: 고객이 "관리자 인증"과 "인증번호가 일치하지 않습니다"라는 문장을 반복적으로 말하며, 로그인 또는 시스템 접근 관련 문제가 발생한 것으로 보입니다. 이는 서비스 이용 중 발생한 문제로, "서비스이용" 카테고리에 해당합니다.
정답: 서비스이용 

J16_S000727 gold=서비스이용 pred=서비스이용 finish=stop tok=51
분석: 고객의 발화는 "네 알겠습니다 감사합니다"로, 단순히 확인 및 감사 표현만 포함되어 있으며, 구체적인 의도나 요청 사항이 없음.
정답: 서비스이용 

J16_S000746 gold=서비스이용 pred=서비스이용 finish=stop tok=57
분석: 고객은 인터넷 강의 시청 중 서버 오류로 강의 재생이 되지 않는 문제를 설명하며, 서비스 이용 시 발생한 기술적 문제를 해결해 달라고 요청하고 있습니다.
정답: 서비스이용 



In [10]:
# think-off 250
rows = []
t0 = time.time()
for i, cid in enumerate(manifest.call_id.unique()):
    r = predict_off(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if (i+1) % 25 == 0: print(f"{i+1}/250 ({time.time()-t0:.0f}s)")
df_off = pd.DataFrame(rows)
df_off.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_def_thinkoff.parquet")
p = df_off.dropna(subset=["pred"])
print(f"\nacc: {(p.pred==p.gold).mean():.3f}, 파싱실패: {df_off.pred.isna().sum()}, length: {(df_off.finish=='length').sum()}")
print("예측:\n", df_off.pred.value_counts())

25/250 (231s)
50/250 (543s)
75/250 (830s)
100/250 (1110s)
125/250 (1385s)
150/250 (1652s)
175/250 (1919s)
200/250 (2236s)
225/250 (2559s)
250/250 (2852s)

acc: 0.546, 파싱실패: 1, length: 3
예측:
 pred
환불요청     69
배송확인     52
서비스이용    43
교환반품     28
불만제기     23
주문취소     19
구매진행     15
Name: count, dtype: int64


In [12]:
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

d = df_off.dropna(subset=["pred"]).copy()
CATS = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

# macro-F1 + 불만 지표
macro = f1_score(d.gold, d.pred, labels=CATS, average="macro", zero_division=0)
f1_b = f1_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0)
p_b = precision_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0)
r_b = recall_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0)
print(f"macro-F1: {macro:.3f}")
print(f"불만 F1: {f1_b:.3f} / P: {p_b:.3f} / R: {r_b:.3f}")

# 불만→환불 셀 (그 18셀)
cm = confusion_matrix(d.gold, d.pred, labels=CATS)
gi, pi = CATS.index("불만제기"), CATS.index("환불요청")
print(f"\n불만→환불 셀: {cm[gi][pi]}  (GPT-B는 19)")

# gold=불만제기 행 전체
print("\ngold=불만제기 → 예측 분포:")
for j, c in enumerate(CATS):
    if cm[gi][j] > 0: print(f"  {c}: {cm[gi][j]}")

macro-F1: 0.450
불만 F1: 0.356 / P: 0.565 / R: 0.260

불만→환불 셀: 16  (GPT-B는 19)

gold=불만제기 → 예측 분포:
  환불요청: 16
  주문취소: 1
  불만제기: 13
  배송확인: 12
  교환반품: 6
  구매진행: 1
  서비스이용: 1


In [13]:
for _, r in df_off.head(10).iterrows():
    print(r.pred, "|", r.raw[:120])

구매진행 | 분석: 고객은 "엠베스티"와 "엘리아이"라는 단어를 언급하며, "이거 이재용 봐요"라고 말해 제품 관련 문의를 하고 있는 것으로 보입니다. 이는 구매 진행 과정에서의 문의로 판단됩니다.
정답: 구매진행
서비스이용 | 분석: 고객은 과거에 기기 변경을 여러 번 했으며, 이번에 다시 기기 변경을 시도했으나 로그인 문제가 발생해 진행이 어려워졌다고 설명하고 있습니다. 이는 기기 변경 과정에서 발생한 로그인 문제로 인한 서비스 이용 장
서비스이용 | 분석: 고객이 "어제처럼 눌러요"라고 말하며, 관리자 인증이 필요하다는 점을 언급하고, 인증번호가 일치하지 않는다는 내용을 전달하며, 전용탭 폐쇄 및 초기화 관련 문제를 제기하고 있습니다. 이는 시스템 사용 중 발생
서비스이용 | 분석: 고객의 발화는 "네 알겠습니다 감사합니다"로, 명확한 의도를 나타내지 않으며 단순히 감사 표현만 포함되어 있습니다. 이는 통화의 종료를 알리는 말로 보이나, 의도 분류를 위해 추가 정보가 필요합니다.
정답: 
서비스이용 | 분석: 고객은 인터넷 강의 시청 중 서버 오류로 강의 재생이 되지 않는 문제를 설명하며, 서비스 이용 시 발생한 기술적 문제를 해결해 달라고 요청하고 있습니다.
정답: 서비스이용
서비스이용 | 분석: 고객은 서비스 등록 기간 정보가 5일 이상 변경되었다고 문의하며, 등록한 지 2일밖에 안 됐음에도 불구하고 문제가 발생했다고 주장하고 있습니다. 이는 서비스 이용 관련 문제로 보이며, 등록 정보 변경에 대한 
서비스이용 | 분석: 고객이 "지금 괜찮은가요"라고 물어보며, 이전 발화에서 "디비" 관련 문제와 "정지가 됐다"는 내용이 있었으나, 현재 상태 확인을 위해 문의하는 것으로 보임.
정답: 서비스이용
불만제기 | 분석: 고객은 "20분 전에 그 계획이 중복됐다고 막 빨리 끊어서 삭제해달라고 잘못 드렸거든요. 그래서 삭제해준다고 했는데 삭제 안 돼 가지고...". 이는 계획 중복으로 인해 삭제 요청이 이루어졌으나 처리되지 않아
서비스이용 | 분석: 고객은 기

In [16]:
d = df_off.dropna(subset=["pred"])
from sklearn.metrics import f1_score
print("파싱성공만 macro-F1:", round(f1_score(d.gold, d.pred, labels=CATS, average="macro", zero_division=0), 3))
print("n:", len(d))

파싱성공만 macro-F1: 0.45
n: 249


### think - on

In [17]:
# ===== think-ON: 청크 저장 =====
import time, librosa, pandas as pd, os
from vllm import SamplingParams

sampling_on = SamplingParams(temperature=0.0, max_tokens=3072)
CKPT = "/content/drive/MyDrive/audio_seg_2/qwen3_30b_def_thinkon.parquet"

# build_prompt는 think 켜진 버전 (assistant 줄이 "<|im_start|>assistant\n"으로 끝, <think></think> 안 닫음)
def build_prompt_on(n_audio):
    audio_tags = "".join(f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|>\n" for i in range(1, n_audio+1))
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 고객의 초반 발화 음성을 듣고 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{audio_tags}\n"
        "위 발화들을 듣고 고객 의도를 분류하세요.\n핵심 근거를 간단히 분석한 뒤 정답을 쓰세요.\n형식:\n분석: (간단히)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def predict_on(call_id):
    wavs = get_call_wavs(call_id, max_utterances=5)
    if not wavs:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(w, sr=16000, mono=True)[0], 16000) for w in wavs]
    out = llm.generate([{"prompt": build_prompt_on(len(audios)), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_on, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text[-400:], "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# 이미 한 것 로드 (재개)
done = set()
rows = []
if os.path.exists(CKPT):
    prev = pd.read_parquet(CKPT); rows = prev.to_dict("records"); done = set(prev.call_id)
    print(f"재개: {len(done)}개 완료됨")

t0 = time.time()
all_ids = list(manifest.call_id.unique())
for i, cid in enumerate(all_ids):
    if cid in done: continue
    r = predict_on(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if len(rows) % 25 == 0:   # 25개마다 저장
        pd.DataFrame(rows).to_parquet(CKPT)
        print(f"{len(rows)}/250 저장 ({time.time()-t0:.0f}s)")
pd.DataFrame(rows).to_parquet(CKPT)
print("완료 저장:", len(rows))

100/250 저장 (12172s)
125/250 저장 (14861s)
150/250 저장 (17635s)
175/250 저장 (20861s)
200/250 저장 (24243s)
225/250 저장 (27540s)
250/250 저장 (30707s)
완료 저장: 250


In [6]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

CKPT = "/content/drive/MyDrive/audio_seg_2/qwen3_30b_def_thinkon.parquet"
CATEGORIES = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

df = pd.read_parquet(CKPT)
print(f"저장된 콜: {len(df)}/250")

# gold 없으면 붙이기 (혹시 저장 시 누락 대비)
if "gold" not in df.columns or df.gold.isna().any():
    gold = pd.read_parquet("/content/drive/MyDrive/audio_seg_2/gold_actual_batch1_final.parquet")
    gmap = dict(zip(gold.call_id, gold.label))
    df["gold"] = df.call_id.map(gmap)

# 잘림/파싱 현황
print("\nfinish:", df.finish.value_counts().to_dict())
print(f"length(잘림): {(df.finish=='length').sum()}/{len(df)} = {(df.finish=='length').mean()*100:.1f}%")
print(f"pred=None(파싱실패): {df.pred.isna().sum()}")

# 성능 (파싱 성공만)
d = df.dropna(subset=["pred"])
macro = round(f1_score(d.gold, d.pred, labels=CATEGORIES, average="macro", zero_division=0), 3)
f1_b = round(f1_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
p_b = round(precision_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
r_b = round(recall_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
acc = round((d.pred==d.gold).mean(), 3)

print(f"\n=== think-on 성능 (n={len(d)}) ===")
print(f"macro-F1: {macro}")
print(f"불만 F1: {f1_b} / P: {p_b} / R: {r_b}")
print(f"accuracy: {acc}")

# 18셀
cm = confusion_matrix(d.gold, d.pred, labels=CATEGORIES)
gi, pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")
print(f"\n불만→환불 셀: {cm[gi][pi]}  (GPT-B 19, audio-only 16)")

print("\ngold=불만제기 → 예측 분포:")
for j, c in enumerate(CATEGORIES):
    if cm[gi][j] > 0: print(f"  {c}: {cm[gi][j]}")

print("\n예측 분포:\n", df.pred.value_counts())
print("\ngold 분포:\n", df.gold.value_counts())

저장된 콜: 250/250

finish: {'stop': 225, 'length': 25}
length(잘림): 25/250 = 10.0%
pred=None(파싱실패): 18

=== think-on 성능 (n=232) ===
macro-F1: 0.461
불만 F1: 0.378 / P: 0.519 / R: 0.298
accuracy: 0.591

불만→환불 셀: 21  (GPT-B 19, audio-only 16)

gold=불만제기 → 예측 분포:
  환불요청: 21
  불만제기: 14
  배송확인: 3
  교환반품: 7
  구매진행: 2

예측 분포:
 pred
환불요청     91
서비스이용    34
교환반품     28
배송확인     28
불만제기     27
구매진행     14
주문취소     10
Name: count, dtype: int64

gold 분포:
 gold
환불요청     87
서비스이용    51
불만제기     50
배송확인     24
교환반품     19
구매진행     11
주문취소      8
Name: count, dtype: int64


### think-off + 전사텍스트

#### 약한 강도

In [9]:
# ===== 전사+원음: 정의 + 전사 텍스트 + 원음 (think-off) =====
import time, librosa, re, pandas as pd
from vllm import SamplingParams

sampling_tx = SamplingParams(temperature=0.0, max_tokens=512)

def get_call_utts(call_id):
    # (wav_path, text) 순서대로 최대 5개
    rows = manifest[manifest.call_id == call_id].sort_values("utt_idx")
    utts = []
    for _, r in rows.iterrows():
        p = os.path.join(AUDIO_ROOT, call_id, os.path.basename(r.wav_path))
        if os.path.exists(p):
            utts.append((p, str(r.text)))
    return utts[:5]

def build_prompt_tx(utts):
    # 발화마다 오디오 + 전사 나란히
    blocks = ""
    for i, (_, txt) in enumerate(utts, 1):
        blocks += f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|> 전사: \"{txt}\"\n"
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 각 발화의 음성(원음)과 전사 텍스트를 함께 참고해 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{blocks}\n"
        "위 발화들의 내용(전사)과 말투(음성)를 종합해 고객 의도를 분류하세요.\n"
        "형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def predict_tx(call_id):
    utts = get_call_utts(call_id)
    if not utts:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(p, sr=16000, mono=True)[0], 16000) for p, _ in utts]
    out = llm.generate([{"prompt": build_prompt_tx(utts), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_tx, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text, "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# 5콜 먼저
for cid in manifest.call_id.unique()[:5]:
    r = predict_tx(cid)
    print(f"{cid} gold={gold_map.get(cid)} pred={r['pred']} finish={r['finish']}")
    print(r["raw"][:180], "\n")

WARNING 08-15 23:03:07 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _triton_mrope_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
J16_S000434 gold=서비스이용 pred=배송확인 finish=stop
분석: 고객은 "엠자로 써 있어요?"라고 물어보며, "엠베스트하고 엘리하이 보이거."라고 언급하며 제품 또는 서비스 관련 정보를 확인하거나 문의하는 것으로 보입니다. 이는 배송 상태나 제품 정보 확인과 관련된 문의로, 배송확인 카테고리에 해당합니다.
정답: 배송확인 

J16_S000633 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 작년부터 기기 변경을 시도했으나 올해 재수를 하면서 패스를 구매한 후 기기 변경이 누적되어 실패했으며, 로그인이 막혀 문제가 발생했다고 설명하며, 이는 시스템 이용 관련 문제로 보인다.
정답: 서비스이용 

J16_S000687 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 관리자 인증 과정에서 인증번호 불일치로 인해 문제를 겪고 있으며, 전용탭 해제 및 초기화 옵션을 확인하며 시스템 사용 관련 문제 해결을 요청하는 것으로 보입니다.
정답: 서비스이용 

J16_S000727 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객이 "아, 네 알겠습니다. 감사합니다."라고 말하며, 음성에서 감사의 뉘앙스와 끝맺음이 강조되어 있으나, 구체적인 요청이나 불만 제기 등은 없어 단순히 대화를 마무리하는 것으로 보임.
정답: 서비스이용 

J16_S000746 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 인터넷 강의 시청 중 서버 오류로 인해 강의 재생이 불가능하다고 반복적으로 설명하며, 이는 시스템 이용 

In [10]:
# 전사+원음 250
rows = []
t0 = time.time()
for i, cid in enumerate(manifest.call_id.unique()):
    r = predict_tx(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if (i+1) % 25 == 0: print(f"{i+1}/250 ({time.time()-t0:.0f}s)")
df_tx = pd.DataFrame(rows)
df_tx.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_transcript.parquet")

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
d = df_tx.dropna(subset=["pred"])
print(f"\nmacro-F1: {round(f1_score(d.gold,d.pred,labels=CATEGORIES,average='macro',zero_division=0),3)}")
print(f"불만 F1: {round(f1_score(d.gold,d.pred,labels=['불만제기'],average='micro',zero_division=0),3)}")
print(f"불만 R: {round(recall_score(d.gold,d.pred,labels=['불만제기'],average='micro',zero_division=0),3)}")
cm = confusion_matrix(d.gold,d.pred,labels=CATEGORIES)
gi,pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")
print(f"불만→환불 셀: {cm[gi][pi]} (GPT-B 19, audio-only 16)")
print(f"acc: {round((d.pred==d.gold).mean(),3)}, 파싱실패: {df_tx.pred.isna().sum()}")

25/250 (223s)
50/250 (456s)
75/250 (683s)
100/250 (921s)
125/250 (1132s)
150/250 (1361s)
175/250 (1586s)
200/250 (1818s)
225/250 (2058s)
250/250 (2285s)

macro-F1: 0.482
불만 F1: 0.444
불만 R: 0.4
불만→환불 셀: 16 (GPT-B 19, audio-only 16)
acc: 0.572, 파싱실패: 0


In [11]:
from sklearn.metrics import confusion_matrix
import pandas as pd
d = df_tx.dropna(subset=["pred"])
cm = confusion_matrix(d.gold, d.pred, labels=CATEGORIES)
cm_df = pd.DataFrame(cm, index=[f"gold_{c}" for c in CATEGORIES], columns=[f"pred_{c}" for c in CATEGORIES])
print(cm_df)
# 가장 큰 오분류 셀 top 8
errors = []
for i, gc in enumerate(CATEGORIES):
    for j, pc in enumerate(CATEGORIES):
        if i != j and cm[i][j] > 0:
            errors.append((cm[i][j], f"{gc}→{pc}"))
print("\n최다 오분류:")
for n, e in sorted(errors, reverse=True)[:8]:
    print(f"  {e}: {n}")

            pred_환불요청  pred_주문취소  pred_불만제기  pred_배송확인  pred_교환반품  pred_구매진행  \
gold_환불요청          52         12          2          5          7          8   
gold_주문취소           2          1          2          1          0          1   
gold_불만제기          16          1         20          9          3          0   
gold_배송확인           1          0          2         19          0          0   
gold_교환반품           1          0          3          5          9          0   
gold_구매진행           0          1          3          0          0          5   
gold_서비스이용          0          1          8          3          1          1   

            pred_서비스이용  
gold_환불요청            1  
gold_주문취소            1  
gold_불만제기            1  
gold_배송확인            2  
gold_교환반품            1  
gold_구매진행            2  
gold_서비스이용          37  

최다 오분류:
  불만제기→환불요청: 16
  환불요청→주문취소: 12
  불만제기→배송확인: 9
  환불요청→구매진행: 8
  서비스이용→불만제기: 8
  환불요청→교환반품: 7
  환불요청→배송확인: 5
  교환반품→배송확인: 5


#### 강한 강도

In [12]:
# ===== 전사+원음 + 경계규칙 강화 (think-off) =====
import time, librosa, re, pandas as pd
from vllm import SamplingParams

sampling_tx = SamplingParams(temperature=0.0, max_tokens=512)

# 기존 정의 + 경계 규칙 추가
CATEGORY_DEF_ENHANCED = CATEGORY_DEF + """

[분류 시 주의 - 헷갈리는 경계 규칙]
1. 불만제기 vs 환불요청/배송확인: 고객이 환불·배송을 언급해도, 말투에 항의·짜증·격앙된 감정이 실려 있고 '문제 제기' 자체가 핵심이면 불만제기입니다. 단순히 절차를 요청하는 차분한 어조면 환불요청/배송확인입니다. 음성의 어조를 반드시 반영하세요.
2. 환불요청 vs 주문취소 vs 교환반품: 돈을 돌려받으려는 것=환불요청, 주문 자체를 없애려는 것=주문취소, 물건을 바꾸거나 되돌려보내는 것=교환반품입니다.
3. 서비스이용 vs 불만제기: 기술적 오류·이용 방법·계정 문제 등 '이용 관련 문의'는 서비스이용입니다. 이용 중 불편을 언급해도 항의가 아니라 해결 문의가 목적이면 서비스이용입니다.
"""

def get_call_utts(call_id):
    rows = manifest[manifest.call_id == call_id].sort_values("utt_idx")
    utts = []
    for _, r in rows.iterrows():
        p = os.path.join(AUDIO_ROOT, call_id, os.path.basename(r.wav_path))
        if os.path.exists(p):
            utts.append((p, str(r.text)))
    return utts[:5]

def build_prompt_tx(utts):
    blocks = ""
    for i, (_, txt) in enumerate(utts, 1):
        blocks += f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|> 전사: \"{txt}\"\n"
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 각 발화의 음성(원음)과 전사 텍스트를 함께 참고해 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF_ENHANCED}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{blocks}\n"
        "위 발화들의 내용(전사)과 말투(음성)를 종합해 고객 의도를 분류하세요.\n"
        "형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def predict_tx(call_id):
    utts = get_call_utts(call_id)
    if not utts:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(p, sr=16000, mono=True)[0], 16000) for p, _ in utts]
    out = llm.generate([{"prompt": build_prompt_tx(utts), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_tx, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text, "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# ---- 5콜 먼저 ----
print("=== 5콜 테스트 ===")
for cid in manifest.call_id.unique()[:5]:
    r = predict_tx(cid)
    print(f"{cid} gold={gold_map.get(cid)} pred={r['pred']} finish={r['finish']}")
    print(r["raw"][:180], "\n")

=== 5콜 테스트 ===
J16_S000434 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객이 "엠자로 써 있어요?"라고 물어보며, "엠베스트하고 엘리하이 보이거"라고 언급하며 제품 정보 확인을 요청하는 것으로 보이며, 전체적으로 차분한 어조로 제품 관련 문의를 진행 중입니다.
정답: 서비스이용 

J16_S000633 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 작년부터 기기 변경을 반복해 누적된 문제로 인해 이번에 로그인이 막힌 상황을 설명하며, 기술적 오류로 인한 서비스 이용 장애를 해결하기 위해 문의하는 것으로 보입니다. 말투는 당황스럽고 불편함을 표현하지만, 항의나 불만의 강도가 높지 않고 기술적 문제 해결을 위한 문의로 보입니다.
정답: 서비스이용 

J16_S000687 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 전용탭 해제 및 초기화 절차 관련 문제를 문의하며, 음성에서 긴장감과 당황스러움이 느껴지나, 핵심은 기술적 문제 해결을 위한 도움 요청으로 보입니다.
정답: 서비스이용 

J16_S000727 gold=서비스이용 pred=서비스이용 finish=stop
분석: 전사 텍스트는 "아, 네 알겠습니다. 감사합니다."로, 단순히 확인 및 감사 표현만 포함되어 있으며, 음성의 어조는 차분하고 정중해 추가 요청이나 불만이 없음.
정답: 서비스이용 

J16_S000746 gold=서비스이용 pred=불만제기 finish=stop
분석: 고객은 인터넷 강의 시청 중 서버 오류로 인해 반복적으로 재시도해도 문제 해결되지 않아, 절차적 도움을 요청하는 것이 아니라 오류 발생 자체에 대한 항의와 불만을 표현하고 있습니다.
정답: 불만제기 



In [13]:
# ---- 250 전체 ----
from sklearn.metrics import f1_score, recall_score, confusion_matrix

rows = []
t0 = time.time()
for i, cid in enumerate(manifest.call_id.unique()):
    r = predict_tx(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if (i+1) % 25 == 0: print(f"{i+1}/250 ({time.time()-t0:.0f}s)")

df_tx2 = pd.DataFrame(rows)
df_tx2.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_transcript_enhanced.parquet")

d = df_tx2.dropna(subset=["pred"])
macro = round(f1_score(d.gold, d.pred, labels=CATEGORIES, average="macro", zero_division=0), 3)
f1_b = round(f1_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
r_b = round(recall_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
cm = confusion_matrix(d.gold, d.pred, labels=CATEGORIES)
gi, pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")

print(f"\n=== 경계규칙 강화 결과 (n={len(d)}) ===")
print(f"macro-F1: {macro}   (규칙전 0.482, GPT-B 0.525)")
print(f"불만 F1: {f1_b}   (규칙전 0.444)")
print(f"불만 R: {r_b}   (규칙전 0.400)")
print(f"불만→환불 셀: {cm[gi][pi]}   (규칙전 16, GPT-B 19)")
print(f"acc: {round((d.pred==d.gold).mean(),3)}, 파싱실패: {df_tx2.pred.isna().sum()}")

25/250 (215s)
50/250 (419s)
75/250 (626s)
100/250 (856s)
125/250 (1066s)
150/250 (1278s)
175/250 (1497s)
200/250 (1725s)
225/250 (1952s)
250/250 (2169s)

=== 경계규칙 강화 결과 (n=250) ===
macro-F1: 0.421   (규칙전 0.482, GPT-B 0.525)
불만 F1: 0.563   (규칙전 0.444)
불만 R: 0.76   (규칙전 0.400)
불만→환불 셀: 2   (규칙전 16, GPT-B 19)
acc: 0.516, 파싱실패: 0


In [15]:
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

# 강한규칙 결과 로드
df_strong = pd.read_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_transcript_enhanced.parquet")
d = df_strong.dropna(subset=["pred"])
cm = pd.DataFrame(confusion_matrix(d.gold, d.pred, labels=CATEGORIES),
                  index=[f"g_{c}" for c in CATEGORIES], columns=[f"p_{c}" for c in CATEGORIES])
print(cm)

# 클래스별 F1 — 어디가 무너졌나
print("\n클래스별 F1 (강한규칙):")
for c in CATEGORIES:
    f = f1_score(d.gold, d.pred, labels=[c], average="micro", zero_division=0)
    print(f"  {c}: {round(f,3)}")

# 불만으로 잘못 온 것들 (과예측 확인)
false_불만 = d[(d.pred=="불만제기") & (d.gold!="불만제기")]
print(f"\n불만 과예측(gold≠불만인데 불만 예측): {len(false_불만)}개")
print(false_불만.gold.value_counts())

         p_환불요청  p_주문취소  p_불만제기  p_배송확인  p_교환반품  p_구매진행  p_서비스이용
g_환불요청       29      14      19       4       7      12        2
g_주문취소        0       2       3       1       0       1        1
g_불만제기        2       0      38       5       3       0        2
g_배송확인        1       0       6      15       0       0        2
g_교환반품        0       0       6       6       6       0        1
g_구매진행        0       0       4       2       1       2        2
g_서비스이용       0       1       9       3       1       0       37

클래스별 F1 (강한규칙):
  환불요청: 0.487
  주문취소: 0.16
  불만제기: 0.563
  배송확인: 0.5
  교환반품: 0.324
  구매진행: 0.154
  서비스이용: 0.755

불만 과예측(gold≠불만인데 불만 예측): 47개
gold
환불요청     19
서비스이용     9
교환반품      6
배송확인      6
구매진행      4
주문취소      3
Name: count, dtype: int64


#### 강한강도(2단계)

In [16]:
# ===== 2단계: 불만 예측된 것만 균형 프롬프트로 재분류 =====
import time, librosa, pandas as pd
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# 1단계 = 강한규칙 결과 (df_strong 이미 로드됨)
# 균형 프롬프트(규칙 없음) build 함수 — 재분류용
def build_prompt_balanced(utts):
    blocks = ""
    for i, (_, txt) in enumerate(utts, 1):
        blocks += f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|> 전사: \"{txt}\"\n"
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 각 발화의 음성(원음)과 전사 텍스트를 함께 참고해 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{blocks}\n"
        "위 발화들의 내용(전사)과 말투(음성)를 종합해 고객 의도를 분류하세요.\n"
        "형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def reclassify(call_id):
    utts = get_call_utts(call_id)
    if not utts: return None
    audios = [(librosa.load(p, sr=16000, mono=True)[0], 16000) for p, _ in utts]
    out = llm.generate([{"prompt": build_prompt_balanced(utts), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_tx, use_tqdm=False)
    return parse_pred(out[0].outputs[0].text)

# 2단계 적용: 1단계에서 불만으로 예측된 것만 재분류
df2 = df_strong.copy()
불만_ids = df2[df2.pred=="불만제기"].call_id.tolist()
print(f"불만 예측 {len(불만_ids)}개 재분류 중...")

t0 = time.time()
final_pred = {}
for i, cid in enumerate(불만_ids):
    rp = reclassify(cid)
    final_pred[cid] = rp if rp else "불만제기"  # 재분류 실패시 불만 유지
    if (i+1) % 10 == 0: print(f"  {i+1}/{len(불만_ids)} ({time.time()-t0:.0f}s)")

# 최종 예측 = 불만 아니었던 건 그대로, 불만이었던 건 재분류 결과
df2["pred_2stage"] = df2.apply(lambda r: final_pred.get(r.call_id, r.pred), axis=1)

# 평가
d = df2.dropna(subset=["pred_2stage"])
macro = round(f1_score(d.gold, d.pred_2stage, labels=CATEGORIES, average="macro", zero_division=0), 3)
f1_b = round(f1_score(d.gold, d.pred_2stage, labels=["불만제기"], average="micro", zero_division=0), 3)
r_b = round(recall_score(d.gold, d.pred_2stage, labels=["불만제기"], average="micro", zero_division=0), 3)
cm = confusion_matrix(d.gold, d.pred_2stage, labels=CATEGORIES)
gi, pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")

print(f"\n=== 2단계 결과 ===")
print(f"macro-F1: {macro}   (1단계 강한규칙 0.421 / 규칙없음 0.482 / GPT-B 0.525)")
print(f"불만 F1: {f1_b}   (강한규칙 0.563)")
print(f"불만 R: {r_b}   (강한규칙 0.76)")
print(f"불만→환불 셀: {cm[gi][pi]}   (강한규칙 2)")
df2.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_2stage.parquet")

불만 예측 85개 재분류 중...
  10/85 (79s)
  20/85 (158s)
  30/85 (230s)
  40/85 (319s)
  50/85 (392s)
  60/85 (476s)
  70/85 (545s)
  80/85 (614s)

=== 2단계 결과 ===
macro-F1: 0.456   (1단계 강한규칙 0.421 / 규칙없음 0.482 / GPT-B 0.525)
불만 F1: 0.482   (강한규칙 0.563)
불만 R: 0.4   (강한규칙 0.76)
불만→환불 셀: 14   (강한규칙 2)


#### 강한강도(2단계, v2)


In [17]:
# ===== 2단계 v2: 불만 예측된 것만 이진 재확인 =====
import time, librosa, pandas as pd, re
from sklearn.metrics import f1_score, recall_score, confusion_matrix

def build_prompt_binary(utts):
    blocks = ""
    for i, (_, txt) in enumerate(utts, 1):
        blocks += f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|> 전사: \"{txt}\"\n"
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 통화를 분석합니다. 고객이 '진짜 불만·항의'를 하는지, 아니면 '차분한 단순 요청/문의'인지 구분합니다.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{blocks}\n"
        "이 고객이 명백히 분노·항의·강한 불만을 표출하며 문제를 따지는 것이 주된 목적입니까?\n"
        "음성의 어조(격앙·짜증)를 반드시 반영하세요. 환불·배송을 요청해도 어조가 차분하면 '단순요청'입니다.\n"
        "형식:\n판단: (불만 또는 단순요청)\n"
        "단순요청이면 실제 의도 카테고리도: 카테고리: (환불요청/주문취소/배송확인/교환반품/구매진행/서비스이용 중 하나)\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def binary_recheck(call_id):
    utts = get_call_utts(call_id)
    if not utts: return "불만제기"
    audios = [(librosa.load(p, sr=16000, mono=True)[0], 16000) for p, _ in utts]
    out = llm.generate([{"prompt": build_prompt_binary(utts), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_tx, use_tqdm=False)
    txt = out[0].outputs[0].text
    # "판단: 불만"이면 불만 유지
    m = re.search(r"판단\s*[:：]\s*([가-힣]+)", txt)
    if m and "불만" in m.group(1):
        return "불만제기"
    # 단순요청이면 카테고리 뽑기
    m2 = re.search(r"카테고리\s*[:：]\s*([가-힣]+)", txt)
    if m2 and m2.group(1).strip() in CATEGORIES:
        return m2.group(1).strip()
    return "불만제기"  # 못 뽑으면 불만 유지(보수적)

# 강한규칙에서 불만 예측된 것만 이진 재확인
df3 = df_strong.copy()
불만_ids = df3[df3.pred=="불만제기"].call_id.tolist()
print(f"불만 예측 {len(불만_ids)}개 이진 재확인 중...")

t0 = time.time()
rechecked = {}
for i, cid in enumerate(불만_ids):
    rechecked[cid] = binary_recheck(cid)
    if (i+1) % 10 == 0: print(f"  {i+1}/{len(불만_ids)} ({time.time()-t0:.0f}s)")

df3["pred_v2"] = df3.apply(lambda r: rechecked.get(r.call_id, r.pred), axis=1)

d = df3.dropna(subset=["pred_v2"])
macro = round(f1_score(d.gold, d.pred_v2, labels=CATEGORIES, average="macro", zero_division=0), 3)
f1_b = round(f1_score(d.gold, d.pred_v2, labels=["불만제기"], average="micro", zero_division=0), 3)
r_b = round(recall_score(d.gold, d.pred_v2, labels=["불만제기"], average="micro", zero_division=0), 3)
cm = confusion_matrix(d.gold, d.pred_v2, labels=CATEGORIES)
gi, pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")

print(f"\n=== 2단계 v2 (이진 재확인) ===")
print(f"macro-F1: {macro}   (강한규칙 0.421 / 규칙없음 0.482 / GPT-B 0.525)")
print(f"불만 F1: {f1_b}   (강한규칙 0.563 / 규칙없음 0.444)")
print(f"불만 R: {r_b}   (강한규칙 0.76)")
print(f"불만→환불 셀: {cm[gi][pi]}   (강한규칙 2)")
# 재확인으로 몇 개가 불만→다른걸로 바뀌었나
changed = sum(1 for cid in 불만_ids if rechecked[cid] != "불만제기")
print(f"불만→다른클래스 전환: {changed}/{len(불만_ids)}")

불만 예측 85개 이진 재확인 중...
  10/85 (44s)
  20/85 (83s)
  30/85 (129s)
  40/85 (198s)
  50/85 (289s)
  60/85 (398s)
  70/85 (436s)
  80/85 (563s)

=== 2단계 v2 (이진 재확인) ===
macro-F1: 0.427   (강한규칙 0.421 / 규칙없음 0.482 / GPT-B 0.525)
불만 F1: 0.545   (강한규칙 0.563 / 규칙없음 0.444)
불만 R: 0.6   (강한규칙 0.76)
불만→환불 셀: 6   (강한규칙 2)
불만→다른클래스 전환: 25/85


#### 중간강도 (폐기)

In [14]:
# ===== 전사+원음 + 경계규칙 (중간 강도) =====
import time, librosa, re, pandas as pd
from vllm import SamplingParams
from sklearn.metrics import f1_score, recall_score, confusion_matrix

sampling_tx = SamplingParams(temperature=0.0, max_tokens=512)

# 규칙 1을 좁힘: "명백히 항의·분노가 주 목적일 때만" 불만
CATEGORY_DEF_MID = CATEGORY_DEF + """

[분류 시 주의 - 헷갈리는 경계 규칙]
1. 불만제기는 신중히 판단하세요. 환불·배송·교환을 요청하는 것 자체는 불만제기가 아닙니다. 고객이 명백히 항의하거나, 분노·강한 불만 감정을 표출하며 문제를 따지는 것이 통화의 주된 목적일 때만 불만제기입니다. 요청을 하면서 다소 짜증이 섞인 정도라면, 요청 내용에 맞는 카테고리(환불요청/배송확인 등)로 분류하세요. 음성 어조는 참고하되 과도하게 불만으로 몰지 마세요.
2. 환불요청 vs 주문취소 vs 교환반품: 돈을 돌려받으려는 것=환불요청, 주문 자체를 없애려는 것=주문취소, 물건을 바꾸거나 되돌려보내는 것=교환반품입니다.
3. 서비스이용 vs 불만제기: 기술적 오류·이용 방법·계정 문제 등 '이용 관련 문의'는 서비스이용입니다. 이용 중 불편을 언급해도 항의가 아니라 해결 문의가 목적이면 서비스이용입니다.
"""

def get_call_utts(call_id):
    rows = manifest[manifest.call_id == call_id].sort_values("utt_idx")
    utts = []
    for _, r in rows.iterrows():
        p = os.path.join(AUDIO_ROOT, call_id, os.path.basename(r.wav_path))
        if os.path.exists(p):
            utts.append((p, str(r.text)))
    return utts[:5]

def build_prompt_mid(utts):
    blocks = ""
    for i, (_, txt) in enumerate(utts, 1):
        blocks += f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|> 전사: \"{txt}\"\n"
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 각 발화의 음성(원음)과 전사 텍스트를 함께 참고해 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF_MID}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{blocks}\n"
        "위 발화들의 내용(전사)과 말투(음성)를 종합해 고객 의도를 분류하세요.\n"
        "형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def predict_mid(call_id):
    utts = get_call_utts(call_id)
    if not utts:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None}
    audios = [(librosa.load(p, sr=16000, mono=True)[0], 16000) for p, _ in utts]
    out = llm.generate([{"prompt": build_prompt_mid(utts), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_tx, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text, "finish": r.finish_reason}

# ---- 250 전체 ----
rows = []
t0 = time.time()
for i, cid in enumerate(manifest.call_id.unique()):
    r = predict_mid(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if (i+1) % 25 == 0: print(f"{i+1}/250 ({time.time()-t0:.0f}s)")

df_mid = pd.DataFrame(rows)
df_mid.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_transcript_mid.parquet")

d = df_mid.dropna(subset=["pred"])
macro = round(f1_score(d.gold, d.pred, labels=CATEGORIES, average="macro", zero_division=0), 3)
f1_b = round(f1_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
r_b = round(recall_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
cm = confusion_matrix(d.gold, d.pred, labels=CATEGORIES)
gi, pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")

print(f"\n=== 중간 강도 규칙 결과 (n={len(d)}) ===")
print(f"macro-F1: {macro}   (규칙없음 0.482 / 강한규칙 0.421 / GPT-B 0.525)")
print(f"불만 F1: {f1_b}   (규칙없음 0.444 / 강한규칙 0.563)")
print(f"불만 R: {r_b}   (규칙없음 0.400 / 강한규칙 0.76)")
print(f"불만→환불 셀: {cm[gi][pi]}   (규칙없음 16 / 강한규칙 2)")
print(f"acc: {round((d.pred==d.gold).mean(),3)}")

25/250 (201s)
50/250 (387s)
75/250 (572s)
100/250 (785s)
125/250 (976s)
150/250 (1173s)
175/250 (1380s)
200/250 (1586s)
225/250 (1802s)
250/250 (2017s)

=== 중간 강도 규칙 결과 (n=250) ===
macro-F1: 0.456   (규칙없음 0.482 / 강한규칙 0.421 / GPT-B 0.525)
불만 F1: 0.364   (규칙없음 0.444 / 강한규칙 0.563)
불만 R: 0.24   (규칙없음 0.400 / 강한규칙 0.76)
불만→환불 셀: 16   (규칙없음 16 / 강한규칙 2)
acc: 0.552


#### 강한강도(2단계, v3)

In [19]:
# ============================================================
# 2단계 v3
# Strong = 불만제기, Neutral(df_tx) != 불만제기인 disagreement만
# KEEP / REVERT verifier로 재검증
# ============================================================

import time
import re
import librosa
import pandas as pd
import numpy as np

from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
)

from vllm import SamplingParams


# ============================================================
# 0. 설정
# ============================================================

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]

COMPLAINT = "불만제기"


# 규칙 없음 Qwen 결과
df_neutral = df_tx.copy()

# 강한 규칙 Qwen 결과
# 기존 df_strong 사용
assert "df_strong" in globals(), "df_strong이 없습니다."
assert "df_tx" in globals(), "df_tx가 없습니다."


# 필요한 컬럼 체크
required_cols = {"call_id", "gold", "pred"}

for name, df in [
    ("df_strong", df_strong),
    ("df_tx", df_tx)
]:
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(
            f"{name}에 필요한 컬럼이 없습니다: {missing}"
        )


print(f"df_strong shape: {df_strong.shape}")
print(f"df_tx shape    : {df_tx.shape}")


# ============================================================
# 1. Verifier sampling
# ============================================================

# KEEP / REVERT만 출력하도록 짧고 deterministic하게
sampling_verify = SamplingParams(
    temperature=0.0,
    max_tokens=8,
    stop=["\n", "<|im_end|>"],
)


# ============================================================
# 2. Neutral 클래스별 설명
# ============================================================

BASE_GUIDANCE = {

    "환불요청":
        "환불요청은 결제 금액을 돌려받거나 환급받으려는 요청 자체가 중심입니다. "
        "고객이 환불을 강하게 요구하거나 짜증스러운 어조를 사용하더라도, "
        "별도의 문제 제기나 항의가 뚜렷하지 않다면 환불요청입니다.",

    "주문취소":
        "주문취소는 이미 한 주문을 취소하려는 것이 중심입니다. "
        "취소를 강하게 요구하더라도 서비스나 처리 과정에 대한 별도의 항의가 없다면 주문취소입니다.",

    "배송확인":
        "배송확인은 배송 상태, 위치, 도착 시점 등을 확인하는 것이 중심입니다. "
        "배송이 늦어 답답하거나 급한 목소리를 낸다는 이유만으로 불만제기가 되지는 않습니다.",

    "교환반품":
        "교환반품은 상품의 교환 또는 반품 절차를 진행하려는 것이 중심입니다. "
        "약한 짜증이나 불쾌한 어조가 있더라도 별도의 문제 제기나 항의가 뚜렷하지 않으면 교환반품입니다.",

    "구매진행":
        "구매진행은 상품 구매, 주문, 결제 등 구매 행동을 진행하려는 것이 중심입니다. "
        "말투의 강도보다 실제 발화의 목적을 우선해서 판단합니다.",

    "서비스이용":
        "서비스이용은 서비스 사용법, 기능, 처리 방법, 일반 문의 등이 중심입니다. "
        "서비스 이용 과정에서 답답함이나 불편함을 표현하는 것만으로는 불만제기가 아닙니다.",
}


# ============================================================
# 3. Verifier prompt
# ============================================================

def build_prompt_verifier(utts, base_pred):

    blocks = ""

    for i, (_, txt) in enumerate(utts, 1):

        txt = "" if txt is None else str(txt)

        blocks += (
            f"발화{i}: "
            f"<|audio_start|><|audio_pad|><|audio_end|> "
            f'전사: "{txt}"\n'
        )

    guidance = BASE_GUIDANCE.get(
        base_pred,
        f"{base_pred}와 불만제기의 차이를 실제 고객 의도를 중심으로 판단하세요."
    )

    return (
        "<|im_start|>system\n"

        "당신은 한국어 콜센터 통화의 '불만제기' 라벨을 재검증하는 판정자입니다.\n\n"

        "이 통화는 음성 어조를 강하게 반영한 분류에서 '불만제기'로 탐지되었습니다.\n"
        "그러나 해당 분류는 짜증, 강한 말투, 급한 억양을 불만으로 과대평가할 수 있습니다.\n"
        "따라서 음성과 전사를 함께 보고 실제 불만·항의인지 다시 검증해야 합니다.\n\n"

        "[불만제기 판정 기준]\n"

        "- 불만제기는 고객이 상품, 서비스, 배송, 상담, 처리 과정 등의 문제에 대해 "
        "항의, 비판, 책임 추궁, 명시적인 문제 제기 또는 뚜렷한 불쾌감을 표현하는 경우입니다.\n"

        "- 음성의 격앙, 짜증, 공격적인 억양은 불만을 뒷받침하는 단서가 될 수 있지만 "
        "그 자체만으로 불만제기가 되지는 않습니다.\n"

        "- 단순히 목소리가 크거나 급하거나 짜증스럽다는 이유로 불만제기로 판단하지 마세요.\n"

        "- 환불, 취소, 배송 확인, 교환·반품 등의 요청을 강한 어조로 말하더라도 "
        "별도의 항의나 문제 제기가 뚜렷하지 않다면 원래 요청 카테고리가 더 적절합니다.\n"

        "- 반대로 발화의 표면적인 요청이 환불, 배송 확인 등이라도 "
        "고객이 서비스나 처리 과정의 문제를 따지거나 비판하고 있고 "
        "음성에서도 그 불만이 뚜렷하게 확인된다면 불만제기를 유지합니다.\n"

        "- 핵심은 '고객의 목소리가 부정적인가?'가 아니라 "
        "'불만·항의가 독립적인 의사표현으로 실제 존재하는가?'입니다.\n\n"

        "[현재 비교 대상]\n"
        f"기본 의미 분류 결과: {base_pred}\n"
        f"{base_pred} 판단 기준: {guidance}\n\n"

        "[출력 규칙]\n"

        "- 불만·항의 증거가 충분해서 '불만제기'를 유지해야 하면: KEEP\n"
        f"- 불만제기보다는 '{base_pred}'가 더 적절하면: REVERT\n"

        "- 반드시 KEEP 또는 REVERT 중 하나만 출력하세요.\n"

        "<|im_end|>\n"

        "<|im_start|>user\n"

        f"{blocks}\n"

        f"전사 내용 중심의 기본 분류 결과는 '{base_pred}'입니다.\n"
        "음성 어조를 강하게 반영한 분류 결과는 '불만제기'입니다.\n\n"

        "음성과 전사를 모두 고려했을 때, "
        "기본 의미 분류를 뒤집고 '불만제기'로 판단할 만큼 "
        "충분하고 독립적인 불만·항의 증거가 있습니까?\n\n"

        "KEEP 또는 REVERT 중 하나만 출력하세요."

        "<|im_end|>\n"

        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    )


# ============================================================
# 4. 출력 parser
# ============================================================

def parse_verdict(text):

    if text is None:
        return None

    text = str(text).strip().upper()

    if text.startswith("KEEP"):
        return "KEEP"

    if text.startswith("REVERT"):
        return "REVERT"

    m = re.search(
        r"\b(KEEP|REVERT)\b",
        text
    )

    if m:
        return m.group(1)

    return None


# ============================================================
# 5. 단일 call verifier
# ============================================================

def verify_complaint(call_id, base_pred):

    utts = get_call_utts(call_id)

    if not utts:

        return {
            "verdict": None,
            "raw": "",
            "error": "no_utts",
        }

    try:

        audios = []

        for audio_path, _ in utts:

            wav, _ = librosa.load(
                audio_path,
                sr=16000,
                mono=True,
            )

            audios.append(
                (wav, 16000)
            )

        prompt = build_prompt_verifier(
            utts,
            base_pred
        )

        out = llm.generate(
            [
                {
                    "prompt": prompt,
                    "multi_modal_data": {
                        "audio": audios
                    },
                }
            ],
            sampling_params=sampling_verify,
            use_tqdm=False,
        )

        raw = (
            out[0]
            .outputs[0]
            .text
            .strip()
        )

        verdict = parse_verdict(raw)

        return {
            "verdict": verdict,
            "raw": raw,
            "error": None,
        }

    except Exception as e:

        return {
            "verdict": None,
            "raw": "",
            "error": repr(e),
        }


# ============================================================
# 6. Strong / Neutral merge
# ============================================================

strong = (
    df_strong[
        ["call_id", "gold", "pred"]
    ]
    .copy()
    .rename(
        columns={
            "pred": "pred_strong"
        }
    )
)


neutral = (
    df_tx[
        ["call_id", "gold", "pred"]
    ]
    .copy()
    .rename(
        columns={
            "gold": "gold_neutral",
            "pred": "pred_neutral"
        }
    )
)


df_v3 = strong.merge(
    neutral,
    on="call_id",
    how="left"
)


print(
    f"\nmerge 후 shape: {df_v3.shape}"
)


# ============================================================
# 7. Strong / Neutral gold 일치 여부
# ============================================================

gold_mismatch = (
    df_v3["gold_neutral"].notna()
    &
    (
        df_v3["gold"]
        != df_v3["gold_neutral"]
    )
)


if gold_mismatch.any():

    print(
        f"[WARNING] Strong / Neutral gold 불일치: "
        f"{gold_mismatch.sum()}건"
    )

else:

    print(
        "[OK] Strong / Neutral gold 일치"
    )


# ============================================================
# 8. Disagreement case 정의
#
# Strong = 불만
# Neutral != 불만
#
# 이 케이스에 대해서만 verifier 실행
# ============================================================

df_v3["need_verify"] = (

    (df_v3["pred_strong"] == COMPLAINT)

    &

    (df_v3["pred_neutral"].notna())

    &

    (df_v3["pred_neutral"] != COMPLAINT)
)


n_strong_complaint = (
    df_v3["pred_strong"]
    == COMPLAINT
).sum()


n_both_complaint = (

    (df_v3["pred_strong"] == COMPLAINT)

    &

    (df_v3["pred_neutral"] == COMPLAINT)

).sum()


n_verify = (
    df_v3["need_verify"]
).sum()


print("\n" + "=" * 60)

print("Verifier 대상")

print("=" * 60)

print(
    f"Strong 불만 예측        : "
    f"{n_strong_complaint}"
)

print(
    f"Strong/Neutral 둘 다 불만: "
    f"{n_both_complaint}"
)

print(
    f"충돌 → verifier 대상    : "
    f"{n_verify}"
)


print(
    "\nVerifier 대상의 Neutral prediction 분포:"
)

print(
    df_v3.loc[
        df_v3["need_verify"],
        "pred_neutral"
    ]
    .value_counts()
)


# ============================================================
# 9. Verifier 실행
# ============================================================

verify_results = {}


target_rows = (
    df_v3[
        df_v3["need_verify"]
    ]
    .copy()
)


print("\n" + "=" * 60)

print(
    f"Verifier 실행 시작: "
    f"{len(target_rows)}건"
)

print("=" * 60)


t0 = time.time()


for idx, (_, row) in enumerate(
    target_rows.iterrows(),
    1
):

    cid = row["call_id"]

    base_pred = row["pred_neutral"]


    result = verify_complaint(
        call_id=cid,
        base_pred=base_pred
    )


    verify_results[cid] = result


    if (
        idx % 10 == 0
        or
        idx == len(target_rows)
    ):

        elapsed = (
            time.time()
            - t0
        )


        n_keep = sum(
            x["verdict"] == "KEEP"
            for x in verify_results.values()
        )


        n_revert = sum(
            x["verdict"] == "REVERT"
            for x in verify_results.values()
        )


        n_fail = sum(
            x["verdict"] is None
            for x in verify_results.values()
        )


        print(
            f"{idx}/{len(target_rows)} "
            f"| {elapsed:.0f}s "
            f"| KEEP={n_keep} "
            f"| REVERT={n_revert} "
            f"| FAIL={n_fail}"
        )


# ============================================================
# 10. Verifier 결과 저장
# ============================================================

df_v3["verdict"] = None

df_v3["verifier_raw"] = None

df_v3["verifier_error"] = None


for i, row in df_v3.iterrows():

    cid = row["call_id"]


    if cid not in verify_results:
        continue


    result = verify_results[cid]


    df_v3.at[
        i,
        "verdict"
    ] = result["verdict"]


    df_v3.at[
        i,
        "verifier_raw"
    ] = result["raw"]


    df_v3.at[
        i,
        "verifier_error"
    ] = result["error"]


# ============================================================
# 11. 최종 prediction
# ============================================================

def make_final_pred(row):

    strong_pred = row["pred_strong"]

    neutral_pred = row["pred_neutral"]

    verdict = row["verdict"]


    # ----------------------------------------
    # Strong이 불만이 아니면
    # Strong prediction 그대로
    # ----------------------------------------

    if strong_pred != COMPLAINT:

        return strong_pred


    # ----------------------------------------
    # Strong / Neutral 모두 불만이면 유지
    # ----------------------------------------

    if neutral_pred == COMPLAINT:

        return COMPLAINT


    # ----------------------------------------
    # Neutral 결과 없는 경우
    # ----------------------------------------

    if pd.isna(neutral_pred):

        return COMPLAINT


    # ----------------------------------------
    # Verifier
    # ----------------------------------------

    if verdict == "KEEP":

        return COMPLAINT


    if verdict == "REVERT":

        return neutral_pred


    # ----------------------------------------
    # Parser failure
    #
    # 일단 Strong 유지.
    # 아래에서 fail 건수는 별도 출력.
    # ----------------------------------------

    return COMPLAINT


df_v3["pred_v3"] = (
    df_v3.apply(
        make_final_pred,
        axis=1
    )
)


df_v3["changed"] = (

    df_v3["pred_v3"]

    !=

    df_v3["pred_strong"]

)


# ============================================================
# 12. 평가 함수
# ============================================================

def evaluate_predictions(
    df,
    pred_col,
    title
):

    d = (
        df.dropna(
            subset=[
                "gold",
                pred_col
            ]
        )
        .copy()
    )


    macro = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="macro",
        zero_division=0,
    )


    complaint_f1 = f1_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )


    complaint_recall = recall_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )


    complaint_precision = precision_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )


    cm = confusion_matrix(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
    )


    cm_df = pd.DataFrame(

        cm,

        index=[
            f"g_{x}"
            for x in CATEGORIES
        ],

        columns=[
            f"p_{x}"
            for x in CATEGORIES
        ],
    )


    gi = CATEGORIES.index(
        COMPLAINT
    )

    pi_refund = CATEGORIES.index(
        "환불요청"
    )


    print(
        "\n"
        + "=" * 65
    )

    print(title)

    print(
        "=" * 65
    )


    print(
        f"macro-F1       : "
        f"{macro:.3f}"
    )

    print(
        f"불만 Precision : "
        f"{complaint_precision:.3f}"
    )

    print(
        f"불만 Recall    : "
        f"{complaint_recall:.3f}"
    )

    print(
        f"불만 F1        : "
        f"{complaint_f1:.3f}"
    )

    print(
        f"불만→환불 셀   : "
        f"{cm[gi, pi_refund]}"
    )


    print(
        "\nConfusion Matrix"
    )

    print(
        cm_df
    )


    print(
        "\n클래스별 F1"
    )


    report = classification_report(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        output_dict=True,
        zero_division=0,
    )


    for c in CATEGORIES:

        print(
            f"  {c:<7}: "
            f"{report[c]['f1-score']:.3f}"
        )


    return {

        "macro_f1":
            macro,

        "complaint_precision":
            complaint_precision,

        "complaint_recall":
            complaint_recall,

        "complaint_f1":
            complaint_f1,

        "cm":
            cm,

        "cm_df":
            cm_df,

        "report":
            report,
    }


# ============================================================
# 13. Strong / Neutral / v3 비교
# ============================================================

res_strong = evaluate_predictions(

    df_v3,

    "pred_strong",

    "① Strong rule"
)


res_neutral = evaluate_predictions(

    df_v3,

    "pred_neutral",

    "② Neutral / 규칙 없음 (df_tx)"
)


res_v3 = evaluate_predictions(

    df_v3,

    "pred_v3",

    "③ Strong + Disagreement Verifier"
)


# ============================================================
# 14. Strong의 TP / FP에서 verifier가 무엇을 했는지
# ============================================================

strong_complaint_mask = (

    df_v3["pred_strong"]

    == COMPLAINT
)


strong_tp_mask = (

    strong_complaint_mask

    &

    (
        df_v3["gold"]
        == COMPLAINT
    )
)


strong_fp_mask = (

    strong_complaint_mask

    &

    (
        df_v3["gold"]
        != COMPLAINT
    )
)


strong_tp = (
    strong_tp_mask.sum()
)

strong_fp = (
    strong_fp_mask.sum()
)


# ----------------------------------------
# 잘못된 불만 예측을 제거한 수
# ----------------------------------------

fp_removed_mask = (

    strong_fp_mask

    &

    (
        df_v3["pred_v3"]
        != COMPLAINT
    )
)


fp_removed = (
    fp_removed_mask.sum()
)


# ----------------------------------------
# 원래 맞았던 불만을 잃은 수
# ----------------------------------------

tp_lost_mask = (

    strong_tp_mask

    &

    (
        df_v3["pred_v3"]
        != COMPLAINT
    )
)


tp_lost = (
    tp_lost_mask.sum()
)


tp_kept = (

    strong_tp_mask

    &

    (
        df_v3["pred_v3"]
        == COMPLAINT
    )

).sum()


fp_remaining = (

    strong_fp_mask

    &

    (
        df_v3["pred_v3"]
        == COMPLAINT
    )

).sum()


parse_fail = (

    df_v3["need_verify"]

    &

    df_v3["verdict"].isna()

).sum()


print(
    "\n"
    + "=" * 65
)

print(
    "Verifier 효과"
)

print(
    "=" * 65
)


print(
    f"Strong 원래 불만 TP : "
    f"{strong_tp}"
)

print(
    f"Strong 원래 불만 FP : "
    f"{strong_fp}"
)


print()


print(
    f"TP 유지 : "
    f"{tp_kept}/{strong_tp}"
)


print(
    f"TP 손실 : "
    f"{tp_lost}/{strong_tp}"
)


print(
    f"FP 제거 : "
    f"{fp_removed}/{strong_fp}"
)


print(
    f"FP 잔존 : "
    f"{fp_remaining}/{strong_fp}"
)


print(
    f"Verifier parse/error : "
    f"{parse_fail}"
)


# ============================================================
# 15. REVERT 분석
# ============================================================

reverted = (
    df_v3[
        df_v3["verdict"]
        == "REVERT"
    ]
    .copy()
)


if len(reverted) > 0:

    reverted["revert_correct"] = (

        reverted["pred_neutral"]

        ==

        reverted["gold"]
    )


    print(
        "\n"
        + "=" * 65
    )

    print(
        "REVERT 분석"
    )

    print(
        "=" * 65
    )


    print(
        f"총 REVERT : "
        f"{len(reverted)}"
    )


    print(
        f"REVERT 후 정답 : "
        f"{reverted['revert_correct'].sum()}"
        f"/{len(reverted)} "
        f"({reverted['revert_correct'].mean():.3f})"
    )


    print(
        "\nREVERT 대상 gold 분포"
    )

    print(
        reverted["gold"]
        .value_counts()
    )


    print(
        "\nREVERT 목적지 분포"
    )

    print(
        reverted["pred_neutral"]
        .value_counts()
    )


# ============================================================
# 16. 최종 불만 FP 분포
# ============================================================

remaining_fp = df_v3[

    (df_v3["pred_v3"] == COMPLAINT)

    &

    (df_v3["gold"] != COMPLAINT)

]


print(
    "\n"
    + "=" * 65
)

print(
    "최종 불만 과예측 분포"
)

print(
    "=" * 65
)


print(
    remaining_fp["gold"]
    .value_counts()
)


# ============================================================
# 17. Verifier가 변경한 사례 확인
# ============================================================

changed_df = df_v3[
    df_v3["changed"]
].copy()


print(
    "\n"
    + "=" * 65
)

print(
    f"최종 prediction 변경 사례: "
    f"{len(changed_df)}건"
)

print(
    "=" * 65
)


if len(changed_df) > 0:

    print(
        changed_df[
            [
                "call_id",
                "gold",
                "pred_neutral",
                "pred_strong",
                "verdict",
                "pred_v3"
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 18. 상세 분석용 DataFrame
# ============================================================

analysis_cols = [

    "call_id",

    "gold",

    "pred_neutral",

    "pred_strong",

    "need_verify",

    "verdict",

    "pred_v3",

    "changed",

    "verifier_raw",

    "verifier_error",
]


df_verify_analysis = (

    df_v3[
        analysis_cols
    ]
    .copy()
)


print(
    "\n완료."
)

print(
    "상세 결과: df_verify_analysis"
)

df_strong shape: (250, 6)
df_tx shape    : (250, 6)

merge 후 shape: (250, 5)
[OK] Strong / Neutral gold 일치

Verifier 대상
Strong 불만 예측        : 85
Strong/Neutral 둘 다 불만: 33
충돌 → verifier 대상    : 52

Verifier 대상의 Neutral prediction 분포:
pred_neutral
환불요청     26
배송확인     13
서비스이용     4
교환반품      4
주문취소      4
구매진행      1
Name: count, dtype: int64

Verifier 실행 시작: 52건
10/52 | 4s | KEEP=0 | REVERT=10 | FAIL=0
20/52 | 8s | KEEP=0 | REVERT=20 | FAIL=0
30/52 | 12s | KEEP=0 | REVERT=30 | FAIL=0
40/52 | 16s | KEEP=0 | REVERT=40 | FAIL=0
50/52 | 21s | KEEP=0 | REVERT=50 | FAIL=0
52/52 | 21s | KEEP=0 | REVERT=52 | FAIL=0

① Strong rule
macro-F1       : 0.421
불만 Precision : 0.447
불만 Recall    : 0.760
불만 F1        : 0.563
불만→환불 셀   : 2

Confusion Matrix
         p_환불요청  p_주문취소  p_불만제기  p_배송확인  p_교환반품  p_구매진행  p_서비스이용
g_환불요청       29      14      19       4       7      12        2
g_주문취소        0       2       3       1       0       1        1
g_불만제기        2       0      38       5       3       0  

#### 스코어기반

In [21]:
# ============================================================

# Strong=불만 / Neutral(df_tx)!=불만 disagreement에 대해
# complaint score 0~4 추출 후 threshold sweep
# ============================================================

import time
import re
import librosa
import pandas as pd
import numpy as np

from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
)

from vllm import SamplingParams


# ============================================================
# 0. 설정
# ============================================================

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]

COMPLAINT = "불만제기"

# 규칙 없음 결과
df_neutral = df_tx.copy()

assert "df_strong" in globals()
assert "df_tx" in globals()


# ============================================================
# 1. Sampling
# ============================================================

# 숫자 하나만 출력
sampling_score = SamplingParams(
    temperature=0.0,
    max_tokens=4,
    stop=["\n", "<|im_end|>"],
)


# ============================================================
# 2. 클래스별 비교 기준
# ============================================================

BASE_GUIDANCE = {

    "환불요청":
        "환불요청은 환급이나 결제 취소 자체가 중심입니다. "
        "환불을 강하게 요구하거나 짜증스럽게 말하는 것만으로는 불만제기가 아닙니다.",

    "주문취소":
        "주문취소는 이미 한 주문을 취소하려는 것이 중심입니다. "
        "강한 말투만으로 불만제기가 되지는 않습니다.",

    "배송확인":
        "배송확인은 배송 상태, 위치, 도착 시점 등을 확인하려는 것이 중심입니다. "
        "배송 지연 때문에 답답하거나 급한 어조를 보이는 것만으로는 불만제기가 아닙니다.",

    "교환반품":
        "교환반품은 교환 또는 반품 절차가 중심입니다. "
        "불편함이나 짜증이 조금 있다고 해서 반드시 불만제기는 아닙니다.",

    "구매진행":
        "구매진행은 상품 구매, 주문, 결제 등을 진행하려는 것이 중심입니다.",

    "서비스이용":
        "서비스이용은 서비스 사용법, 기능, 처리 방법, 일반 문의 등이 중심입니다. "
        "서비스 이용 중 답답함을 표현하는 것만으로는 불만제기가 아닙니다.",
}


# ============================================================
# 3. Complaint score prompt
# ============================================================

def build_prompt_score(utts, base_pred):

    blocks = ""

    for i, (_, txt) in enumerate(utts, 1):

        txt = "" if txt is None else str(txt)

        blocks += (
            f"발화{i}: "
            f"<|audio_start|><|audio_pad|><|audio_end|> "
            f'전사: "{txt}"\n'
        )

    guidance = BASE_GUIDANCE.get(
        base_pred,
        f"{base_pred}와 불만제기의 차이를 실제 의도를 중심으로 판단하세요."
    )

    return (
        "<|im_start|>system\n"

        "당신은 한국어 콜센터 고객 발화의 불만·항의 강도를 평가합니다.\n"
        "음성과 전사를 모두 사용하되, 최종 카테고리를 직접 결정하지 말고 "
        "불만·항의 증거의 강도만 평가하세요.\n\n"

        "[불만 강도 점수]\n"

        "0 = 불만·항의 증거가 없음. 일반적인 요청이나 문의.\n"

        "1 = 약한 불편함, 답답함, 짜증은 있으나 "
        "독립적인 항의나 문제 제기로 보기 어려움.\n"

        "2 = 불만 가능성이 있으나 애매함. "
        "부정적 어조 또는 문제 언급은 있으나 "
        "단순 요청/문의와 명확히 구분하기 어려움.\n"

        "3 = 명확한 불만·항의. "
        "고객이 서비스, 상품, 처리 과정 등의 문제를 실제로 따지거나 비판하며 "
        "불쾌감 또는 항의를 분명하게 표현함.\n"

        "4 = 매우 강한 불만·항의. "
        "격앙, 반복적인 항의, 강한 비판, 책임 추궁 등이 명백하게 나타남.\n\n"

        "[중요한 판단 원칙]\n"

        "- 목소리가 크거나 짜증스럽다는 이유만으로 높은 점수를 주지 마세요.\n"

        "- 환불, 취소, 배송 확인 등의 요청을 강하게 말하는 것과 "
        "서비스나 처리 과정에 대해 항의하는 것은 구분해야 합니다.\n"

        "- 반대로 표현이 차분하더라도 문제를 명확하게 비판하고 항의한다면 "
        "높은 점수가 가능합니다.\n"

        "- 음성 어조는 중요한 근거이지만, 전사 내용과 함께 판단하세요.\n\n"

        f"[비교되는 기본 의미 카테고리]\n"
        f"{base_pred}\n"
        f"{guidance}\n\n"

        "[출력 규칙]\n"
        "반드시 0, 1, 2, 3, 4 중 숫자 하나만 출력하세요.\n"

        "<|im_end|>\n"

        "<|im_start|>user\n"

        f"{blocks}\n"

        f"전사 내용 중심의 기본 분류 결과는 '{base_pred}'입니다.\n\n"

        "이 고객의 불만·항의 증거 강도를 0~4 중 하나로 평가하세요.\n"

        "<|im_end|>\n"

        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    )


# ============================================================
# 4. Score parser
# ============================================================

def parse_score(text):

    if text is None:
        return None

    text = str(text).strip()

    # 숫자 하나만 찾기
    m = re.search(r"\b([0-4])\b", text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# 5. 단일 call scoring
# ============================================================

def score_complaint(call_id, base_pred):

    utts = get_call_utts(call_id)

    if not utts:
        return {
            "score": None,
            "raw": "",
            "error": "no_utts",
        }

    try:

        audios = []

        for audio_path, _ in utts:

            wav, _ = librosa.load(
                audio_path,
                sr=16000,
                mono=True
            )

            audios.append(
                (wav, 16000)
            )

        prompt = build_prompt_score(
            utts,
            base_pred
        )

        out = llm.generate(
            [
                {
                    "prompt": prompt,
                    "multi_modal_data": {
                        "audio": audios
                    }
                }
            ],
            sampling_params=sampling_score,
            use_tqdm=False,
        )

        raw = out[0].outputs[0].text.strip()

        score = parse_score(raw)

        return {
            "score": score,
            "raw": raw,
            "error": None,
        }

    except Exception as e:

        return {
            "score": None,
            "raw": "",
            "error": repr(e),
        }


# ============================================================
# 6. Strong + Neutral merge
# ============================================================

strong = (
    df_strong[
        ["call_id", "gold", "pred"]
    ]
    .copy()
    .rename(
        columns={
            "pred": "pred_strong"
        }
    )
)

neutral = (
    df_tx[
        ["call_id", "gold", "pred"]
    ]
    .copy()
    .rename(
        columns={
            "gold": "gold_neutral",
            "pred": "pred_neutral"
        }
    )
)

df_score = strong.merge(
    neutral,
    on="call_id",
    how="left"
)


# ============================================================
# 7. Disagreement 대상
#
# strong = 불만
# neutral != 불만
# ============================================================

df_score["need_score"] = (
    (df_score["pred_strong"] == COMPLAINT)
    &
    (df_score["pred_neutral"].notna())
    &
    (df_score["pred_neutral"] != COMPLAINT)
)

target_rows = df_score[
    df_score["need_score"]
].copy()


print("=" * 60)
print("Complaint score 대상")
print("=" * 60)

print(
    f"Strong 불만 예측: "
    f"{(df_score['pred_strong'] == COMPLAINT).sum()}"
)

print(
    f"Score 대상: "
    f"{len(target_rows)}"
)

print("\nNeutral prediction 분포:")

print(
    target_rows["pred_neutral"]
    .value_counts()
)


# ============================================================
# 8. Score 추출
# ============================================================

score_results = {}

t0 = time.time()

print("\nScoring 시작...")


for idx, (_, row) in enumerate(
    target_rows.iterrows(),
    1
):

    cid = row["call_id"]
    base_pred = row["pred_neutral"]

    result = score_complaint(
        call_id=cid,
        base_pred=base_pred,
    )

    score_results[cid] = result

    if (
        idx % 10 == 0
        or idx == len(target_rows)
    ):

        valid_scores = [
            x["score"]
            for x in score_results.values()
            if x["score"] is not None
        ]

        print(
            f"{idx}/{len(target_rows)} "
            f"| {time.time()-t0:.0f}s "
            f"| valid={len(valid_scores)} "
            f"| fail={idx-len(valid_scores)}"
        )


# ============================================================
# 9. DataFrame에 score 저장
# ============================================================

df_score["complaint_score"] = np.nan
df_score["score_raw"] = None
df_score["score_error"] = None


for i, row in df_score.iterrows():

    cid = row["call_id"]

    if cid not in score_results:
        continue

    r = score_results[cid]

    df_score.at[i, "complaint_score"] = r["score"]
    df_score.at[i, "score_raw"] = r["raw"]
    df_score.at[i, "score_error"] = r["error"]


print("\nScore 분포:")

print(
    df_score.loc[
        df_score["need_score"],
        "complaint_score"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 10. Threshold에 따른 최종 prediction
#
# score >= threshold → 불만 유지
# score < threshold  → neutral prediction으로 REVERT
# ============================================================

def make_pred_with_threshold(row, threshold):

    strong_pred = row["pred_strong"]
    neutral_pred = row["pred_neutral"]

    # Strong 자체가 불만이 아니면 strong 결과 사용
    if strong_pred != COMPLAINT:
        return strong_pred

    # Strong / Neutral 모두 불만이면 유지
    if neutral_pred == COMPLAINT:
        return COMPLAINT

    # Neutral 없는 경우 strong 유지
    if pd.isna(neutral_pred):
        return COMPLAINT

    # score 대상인데 parsing 실패
    if pd.isna(row["complaint_score"]):
        return COMPLAINT

    # 핵심 decision rule
    if row["complaint_score"] >= threshold:
        return COMPLAINT

    return neutral_pred


# ============================================================
# 11. Threshold sweep
# ============================================================

sweep_results = []


# threshold 의미:
#
# 0 → 사실상 전부 complaint 유지
# 1 → score 1~4 complaint
# 2 → score 2~4 complaint
# 3 → score 3~4 complaint
# 4 → score 4만 complaint
# 5 → score 모두 neutral로 revert
#
# 5도 비교를 위해 넣음
# ============================================================

for threshold in [0, 1, 2, 3, 4, 5]:

    pred_col = f"pred_t{threshold}"

    df_score[pred_col] = df_score.apply(
        lambda row: make_pred_with_threshold(
            row,
            threshold
        ),
        axis=1,
    )

    d = df_score.dropna(
        subset=[
            "gold",
            pred_col
        ]
    )

    macro = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="macro",
        zero_division=0,
    )

    complaint_f1 = f1_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )

    complaint_precision = precision_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )

    complaint_recall = recall_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )

    # complaint TP/FP/FN 직접 계산
    tp = (
        (d["gold"] == COMPLAINT)
        &
        (d[pred_col] == COMPLAINT)
    ).sum()

    fp = (
        (d["gold"] != COMPLAINT)
        &
        (d[pred_col] == COMPLAINT)
    ).sum()

    fn = (
        (d["gold"] == COMPLAINT)
        &
        (d[pred_col] != COMPLAINT)
    ).sum()

    sweep_results.append(
        {
            "threshold": threshold,
            "macro_f1": macro,
            "complaint_precision": complaint_precision,
            "complaint_recall": complaint_recall,
            "complaint_f1": complaint_f1,
            "complaint_TP": tp,
            "complaint_FP": fp,
            "complaint_FN": fn,
        }
    )


df_sweep = pd.DataFrame(
    sweep_results
)


print("\n" + "=" * 80)
print("Threshold Sweep")
print("=" * 80)

print(
    df_sweep.round(3)
    .to_string(index=False)
)


# ============================================================
# 12. 각 목적별 best threshold
# ============================================================

best_macro = df_sweep.loc[
    df_sweep["macro_f1"].idxmax()
]

best_complaint = df_sweep.loc[
    df_sweep["complaint_f1"].idxmax()
]


print("\n" + "=" * 80)
print("Best threshold")
print("=" * 80)

print(
    f"Macro-F1 기준 best threshold: "
    f"{int(best_macro['threshold'])}"
)

print(
    f"  macro-F1      = "
    f"{best_macro['macro_f1']:.3f}"
)

print(
    f"  complaint F1  = "
    f"{best_macro['complaint_f1']:.3f}"
)

print(
    f"  complaint R   = "
    f"{best_macro['complaint_recall']:.3f}"
)


print()

print(
    f"불만 F1 기준 best threshold: "
    f"{int(best_complaint['threshold'])}"
)

print(
    f"  macro-F1      = "
    f"{best_complaint['macro_f1']:.3f}"
)

print(
    f"  complaint F1  = "
    f"{best_complaint['complaint_f1']:.3f}"
)

print(
    f"  complaint R   = "
    f"{best_complaint['complaint_recall']:.3f}"
)


# ============================================================
# 13. 원하는 threshold 선택
#
# 우선 macro-F1 best 사용
# ============================================================

BEST_THRESHOLD = int(
    best_macro["threshold"]
)

df_score["pred_final"] = df_score[
    f"pred_t{BEST_THRESHOLD}"
]


print(
    f"\n선택 threshold = "
    f"{BEST_THRESHOLD}"
)


# ============================================================
# 14. 선택된 threshold 최종 confusion matrix
# ============================================================

cm = confusion_matrix(
    df_score["gold"],
    df_score["pred_final"],
    labels=CATEGORIES,
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"g_{x}"
        for x in CATEGORIES
    ],
    columns=[
        f"p_{x}"
        for x in CATEGORIES
    ],
)


print("\n" + "=" * 80)
print(
    f"최종 Confusion Matrix "
    f"(threshold={BEST_THRESHOLD})"
)
print("=" * 80)

print(cm_df)


# ============================================================
# 15. 클래스별 F1
# ============================================================

report = classification_report(
    df_score["gold"],
    df_score["pred_final"],
    labels=CATEGORIES,
    output_dict=True,
    zero_division=0,
)


print("\n클래스별 F1:")

for c in CATEGORIES:

    print(
        f"{c:<8}: "
        f"{report[c]['f1-score']:.3f}"
    )


# ============================================================
# 16. score와 실제 gold 관계 확인
# ============================================================

scored_only = df_score[
    df_score["need_score"]
].copy()


scored_only["is_gold_complaint"] = (
    scored_only["gold"]
    == COMPLAINT
)


print("\n" + "=" * 80)
print("Score × 실제 불만 여부")
print("=" * 80)

print(
    pd.crosstab(
        scored_only["complaint_score"],
        scored_only["is_gold_complaint"],
        margins=True
    )
)


# ============================================================
# 17. score별 실제 불만 비율
# ============================================================

score_quality = (
    scored_only
    .dropna(
        subset=["complaint_score"]
    )
    .groupby("complaint_score")
    .agg(
        n=("call_id", "size"),
        gold_complaint=(
            "is_gold_complaint",
            "sum"
        ),
        complaint_rate=(
            "is_gold_complaint",
            "mean"
        ),
    )
    .reset_index()
)


print("\nScore별 실제 불만 비율")

print(
    score_quality.round(3)
    .to_string(index=False)
)


# ============================================================
# 18. 분석용 최종 DataFrame
# ============================================================

analysis_cols = [
    "call_id",
    "gold",
    "pred_neutral",
    "pred_strong",
    "need_score",
    "complaint_score",
    "pred_final",
    "score_raw",
    "score_error",
]


df_score_analysis = (
    df_score[
        analysis_cols
    ]
    .copy()
)


print("\n완료.")
print("상세 결과: df_score_analysis")
print("threshold 비교: df_sweep")

Complaint score 대상
Strong 불만 예측: 85
Score 대상: 52

Neutral prediction 분포:
pred_neutral
환불요청     26
배송확인     13
서비스이용     4
교환반품      4
주문취소      4
구매진행      1
Name: count, dtype: int64

Scoring 시작...
10/52 | 3s | valid=10 | fail=0
20/52 | 6s | valid=20 | fail=0
30/52 | 10s | valid=30 | fail=0
40/52 | 13s | valid=40 | fail=0
50/52 | 16s | valid=50 | fail=0
52/52 | 17s | valid=52 | fail=0

Score 분포:
complaint_score
0.0     1
2.0    27
3.0    24
Name: count, dtype: int64

Threshold Sweep
 threshold  macro_f1  complaint_precision  complaint_recall  complaint_f1  complaint_TP  complaint_FP  complaint_FN
         0     0.421                0.447              0.76         0.563            38            47            12
         1     0.431                0.452              0.76         0.567            38            46            12
         2     0.431                0.452              0.76         0.567            38            46            12
         3     0.447                0.509      

#### 스코어기반 v2

In [22]:
# ============================================================
# Neutral-first fusion
#
# 기본 prediction = df_tx (Neutral)
# Strong은 complaint candidate detector로만 사용
# score threshold 이상일 때만 Neutral → 불만 override
#
# 모델 inference 다시 돌릴 필요 없음
# 기존 df_score의 complaint_score 그대로 사용
# ============================================================

from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
)
import pandas as pd


# ============================================================
# 1. Neutral-first prediction
# ============================================================

def make_neutral_first_pred(row, threshold):

    neutral_pred = row["pred_neutral"]
    strong_pred = row["pred_strong"]
    score = row["complaint_score"]

    # -----------------------------------------
    # 기본값은 무조건 Neutral
    # -----------------------------------------
    if pd.isna(neutral_pred):
        return strong_pred

    # Neutral도 이미 불만이면 그대로 유지
    if neutral_pred == COMPLAINT:
        return COMPLAINT

    # -----------------------------------------
    # Strong이 불만을 탐지한 경우에만
    # override 후보
    # -----------------------------------------
    if strong_pred == COMPLAINT:

        # score 실패 시 Neutral 유지
        if pd.isna(score):
            return neutral_pred

        # threshold 이상이면 불만으로 override
        if score >= threshold:
            return COMPLAINT

    # 그 외에는 전부 Neutral 유지
    return neutral_pred


# ============================================================
# 2. Threshold sweep
# ============================================================

neutral_first_results = []


for threshold in [0, 1, 2, 3, 4, 5]:

    pred_col = f"pred_nf_t{threshold}"

    df_score[pred_col] = df_score.apply(
        lambda row: make_neutral_first_pred(
            row,
            threshold
        ),
        axis=1,
    )

    d = df_score.dropna(
        subset=["gold", pred_col]
    )

    macro = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="macro",
        zero_division=0,
    )

    complaint_precision = precision_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )

    complaint_recall = recall_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )

    complaint_f1 = f1_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0,
    )

    tp = (
        (d["gold"] == COMPLAINT)
        & (d[pred_col] == COMPLAINT)
    ).sum()

    fp = (
        (d["gold"] != COMPLAINT)
        & (d[pred_col] == COMPLAINT)
    ).sum()

    fn = (
        (d["gold"] == COMPLAINT)
        & (d[pred_col] != COMPLAINT)
    ).sum()

    neutral_first_results.append({
        "threshold": threshold,
        "macro_f1": macro,
        "complaint_precision": complaint_precision,
        "complaint_recall": complaint_recall,
        "complaint_f1": complaint_f1,
        "complaint_TP": tp,
        "complaint_FP": fp,
        "complaint_FN": fn,
    })


df_sweep_nf = pd.DataFrame(
    neutral_first_results
)


print("\n" + "=" * 90)
print("Neutral-first Threshold Sweep")
print("=" * 90)

print(
    df_sweep_nf
    .round(3)
    .to_string(index=False)
)


# ============================================================
# 3. Baseline들과 비교
# ============================================================

print("\nBaseline")
print("-" * 50)

print("Neutral(df_tx)")
print("  macro-F1      = 0.482")
print("  complaint F1  = 0.444")
print("  complaint R   = 0.400")

print("\nStrong")
print("  macro-F1      = 0.421")
print("  complaint F1  = 0.563")
print("  complaint R   = 0.760")

print("\nGPT")
print("  macro-F1      = 0.525")
print("  complaint F1  = 0.281")
print("  complaint R   = 0.180")


# ============================================================
# 4. Best threshold
# ============================================================

best_macro_nf = df_sweep_nf.loc[
    df_sweep_nf["macro_f1"].idxmax()
]

best_complaint_nf = df_sweep_nf.loc[
    df_sweep_nf["complaint_f1"].idxmax()
]


print("\n" + "=" * 90)
print("Best Neutral-first Threshold")
print("=" * 90)

print(
    f"Macro-F1 기준: threshold="
    f"{int(best_macro_nf['threshold'])}"
)

print(
    f"  macro-F1     = "
    f"{best_macro_nf['macro_f1']:.3f}"
)

print(
    f"  complaint F1 = "
    f"{best_macro_nf['complaint_f1']:.3f}"
)

print(
    f"  complaint R  = "
    f"{best_macro_nf['complaint_recall']:.3f}"
)


print()

print(
    f"Complaint-F1 기준: threshold="
    f"{int(best_complaint_nf['threshold'])}"
)

print(
    f"  macro-F1     = "
    f"{best_complaint_nf['macro_f1']:.3f}"
)

print(
    f"  complaint F1 = "
    f"{best_complaint_nf['complaint_f1']:.3f}"
)

print(
    f"  complaint R  = "
    f"{best_complaint_nf['complaint_recall']:.3f}"
)


# ============================================================
# 5. Macro-F1 best 선택
# ============================================================

BEST_NF_THRESHOLD = int(
    best_macro_nf["threshold"]
)

df_score["pred_nf_final"] = df_score[
    f"pred_nf_t{BEST_NF_THRESHOLD}"
]


# ============================================================
# 6. 최종 confusion matrix
# ============================================================

cm_nf = confusion_matrix(
    df_score["gold"],
    df_score["pred_nf_final"],
    labels=CATEGORIES,
)

cm_nf_df = pd.DataFrame(
    cm_nf,
    index=[
        f"g_{x}"
        for x in CATEGORIES
    ],
    columns=[
        f"p_{x}"
        for x in CATEGORIES
    ],
)


print("\n" + "=" * 90)
print(
    f"Neutral-first 최종 Confusion Matrix "
    f"(threshold={BEST_NF_THRESHOLD})"
)
print("=" * 90)

print(cm_nf_df)


# ============================================================
# 7. 클래스별 F1
# ============================================================

report_nf = classification_report(
    df_score["gold"],
    df_score["pred_nf_final"],
    labels=CATEGORIES,
    output_dict=True,
    zero_division=0,
)


print("\n클래스별 F1")

for c in CATEGORIES:

    print(
        f"{c:<8}: "
        f"{report_nf[c]['f1-score']:.3f}"
    )


Neutral-first Threshold Sweep
 threshold  macro_f1  complaint_precision  complaint_recall  complaint_f1  complaint_TP  complaint_FP  complaint_FN
         0     0.467                0.435              0.80         0.563            40            52            10
         1     0.477                0.440              0.80         0.567            40            51            10
         2     0.477                0.440              0.80         0.567            40            51            10
         3     0.490                0.484              0.62         0.544            31            33            19
         4     0.482                0.500              0.40         0.444            20            20            30
         5     0.482                0.500              0.40         0.444            20            20            30

Baseline
--------------------------------------------------
Neutral(df_tx)
  macro-F1      = 0.482
  complaint F1  = 0.444
  complaint R   = 0.400

Strong
 

### 카테고리 개선 시

In [26]:
# ============================================================
# Qwen3-Omni Zero-shot 재실험
# 변경점: 카테고리 정의 + 경계 규칙 개선
#
# 입력: audio + transcript (초반 5발화)
# 평가: macro-F1 / class F1 / confusion matrix
#
# 기존 객체 필요:
#   - llm
#   - get_call_utts(call_id)
#   - df_tx  (columns: call_id, gold, pred)
# ============================================================

import time
import re
import librosa
import pandas as pd
import numpy as np

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

from vllm import SamplingParams


# ============================================================
# 0. 기본 설정
# ============================================================

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]

COMPLAINT = "불만제기"

assert "df_tx" in globals(), "기존 결과 df_tx가 없습니다."
assert "llm" in globals(), "llm이 없습니다."
assert "get_call_utts" in globals(), "get_call_utts()가 없습니다."


# ============================================================
# 1. Sampling
#
# 가능하면 기존 실험의 sampling_tx를 그대로 사용.
# 그래야 prompt 정의만 바뀐 실험이 됨.
# ============================================================

if "sampling_tx" in globals():

    sampling_redefined = sampling_tx
    print("[INFO] 기존 sampling_tx 그대로 사용")

else:

    sampling_redefined = SamplingParams(
        temperature=0.0,
        max_tokens=16,
    )

    print("[INFO] sampling_tx가 없어 deterministic 설정 사용")


# ============================================================
# 2. 개선된 카테고리 정의 Prompt
# ============================================================

def build_prompt_redefined(utts):

    blocks = ""

    for i, (_, txt) in enumerate(utts, 1):

        txt = "" if txt is None else str(txt)

        blocks += (
            f"발화{i}: "
            f"<|audio_start|><|audio_pad|><|audio_end|> "
            f'전사: "{txt}"\n'
        )

    return (
        "<|im_start|>system\n"

        "당신은 한국어 교육/교재 콜센터의 고객 문의를 분류하는 전문가입니다.\n"
        "고객의 음성과 전사를 함께 보고 아래 7개 카테고리 중 "
        "가장 적절한 하나를 선택하세요.\n\n"

        "단순히 특정 단어가 등장했다는 이유로 분류하지 말고, "
        "고객이 통화에서 최종적으로 무엇을 해결하려는지를 기준으로 판단하세요.\n\n"


        # ----------------------------------------------------
        # 환불요청
        # ----------------------------------------------------

        "[1. 환불요청]\n"
        "이미 결제·발송·이용 등이 진행된 거래에서 "
        "금전 반환이 핵심 목적인 경우입니다.\n"
        "환불, 카드결제 취소, 금액 반환, 반송 후 환급 등이 주요 단서입니다.\n"
        "고객의 최종 관심이 '돈을 돌려받는 것'이라면 환불요청을 우선합니다.\n\n"


        # ----------------------------------------------------
        # 주문취소
        # ----------------------------------------------------

        "[2. 주문취소]\n"
        "상품이나 서비스가 본격적으로 이행되기 전에 "
        "기존 주문 자체를 없애는 것이 핵심 목적인 경우입니다.\n"
        "잘못 주문했거나, 다른 상품과 함께 다시 주문하거나, "
        "배송비 절약을 위해 재주문하려는 경우 등이 대표적입니다.\n"
        "고객의 관심이 환급 금액보다는 '현재 주문을 없애고 다시 주문하는 것'에 있으면 "
        "주문취소를 우선합니다.\n\n"


        # ----------------------------------------------------
        # 불만제기
        # ----------------------------------------------------

        "[3. 불만제기]\n"
        "표면적으로 환불·배송·반품 등을 문의하더라도, "
        "실제 핵심 목적이 처리 지연, 반복 문의, 약속 불이행, 연락 두절, "
        "상담 대응이나 서비스 문제를 따지고 해결을 요구하는 경우입니다.\n"
        "반복해서 연락했음, 오래 기다렸음, 연락이 오기로 했는데 오지 않음, "
        "처리가 계속 안 됨, 즉시 해결을 요구함 등이 중요한 단서입니다.\n"
        "음성의 격앙·짜증·불쾌한 어조는 불만을 뒷받침하는 보조 근거로 사용하세요.\n"
        "단, 단순히 목소리가 크거나 짜증스럽다는 이유만으로 "
        "불만제기로 분류하지 마세요.\n\n"


        # ----------------------------------------------------
        # 배송확인
        # ----------------------------------------------------

        "[4. 배송확인]\n"
        "상품이 언제 도착하는지, 어디에 있는지, 발송되었는지, "
        "송장번호나 배송주소가 무엇인지 등 "
        "배송 상태 또는 배송 소요일 확인이 핵심인 경우입니다.\n"
        "아직 구매하지 않았더라도 "
        "'지금 주문하면 언제 받을 수 있는가'처럼 배송 소요일을 묻는다면 "
        "배송확인에 포함합니다.\n\n"


        # ----------------------------------------------------
        # 교환반품
        # ----------------------------------------------------

        "[5. 교환반품]\n"
        "수령한 상품의 파본, 오배송, 누락, 훼손 등 "
        "상품 자체의 문제 때문에 물건을 교환하거나 반품하는 것이 핵심인 경우입니다.\n"
        "파본, 잘못 배송된 상품, 구성품이나 답지 누락, 상품 손상, "
        "교환, 반송 등이 중요한 단서입니다.\n"
        "금전 반환보다 '물건을 바꾸거나 돌려보내는 것'이 중심이면 "
        "교환반품을 우선합니다.\n\n"


        # ----------------------------------------------------
        # 구매진행
        # ----------------------------------------------------

        "[6. 구매진행]\n"
        "상품이나 강의의 구매·결제를 완료하는 것이 최종 목적인 경우입니다.\n"
        "결제 오류, 가상계좌 문제, 구매 버튼이나 구매 자격 문제, "
        "보안 프로그램 오류 등도 "
        "결국 결제를 완료하기 위해 해결하려는 문제라면 구매진행입니다.\n"
        "기술적인 문제라고 해서 자동으로 서비스이용으로 분류하지 마세요.\n\n"


        # ----------------------------------------------------
        # 서비스이용
        # ----------------------------------------------------

        "[7. 서비스이용]\n"
        "이미 구매·등록된 서비스나 콘텐츠를 사용하는 과정에서 발생한 "
        "문제나 일반적인 이용 문의입니다.\n"
        "기기등록, 수강, 이용제한, 기기변경, 콘텐츠 이용, "
        "서비스 기능이나 사용 방법 등이 대표적입니다.\n"
        "구매·결제를 완료하려는 문제가 아니라 "
        "구매 이후 서비스를 사용하는 문제가 핵심일 때 선택하세요.\n\n"


        # ====================================================
        # 경계 규칙
        # ====================================================

        "[중요한 경계 규칙]\n\n"

        "① 주문취소 vs 환불요청\n"
        "- 주문을 잘못해서 다시 주문하려 함\n"
        "- 다른 상품과 묶어서 다시 주문하려 함\n"
        "- 배송비 때문에 기존 주문을 없애려 함\n"
        "→ 주문취소를 우선합니다.\n\n"

        "이미 발송·이용되었거나, 환불·금액 반환·카드취소 등 "
        "돈을 돌려받는 것이 명시적인 목적이면 환불요청을 우선합니다.\n\n"

        "고객 발화만으로 출고 여부나 실제 환불 처리 여부를 알 수 없다면 "
        "존재하지 않는 정보를 추측하지 말고 "
        "현재 고객이 명시적으로 표현한 목적을 기준으로 판단하세요.\n\n"


        "② 불만제기 vs 기타 요청\n"
        "환불, 취소, 배송, 반품을 강한 어조로 요구하는 것만으로 "
        "불만제기가 되는 것은 아닙니다.\n"
        "기존 문제의 미처리, 반복 문의, 약속 불이행, 연락 지연, "
        "상담 대응 문제 등을 독립적으로 따지고 항의하는 경우 "
        "불만제기를 우선합니다.\n\n"


        "③ 교환반품 vs 환불요청\n"
        "받은 물건의 파본·오배송·누락·훼손 문제를 해결하기 위해 "
        "상품을 바꾸거나 돌려보내는 것이 중심이면 교환반품입니다.\n"
        "금액을 돌려받는 것이 최종 목적이면 환불요청입니다.\n\n"


        "④ 배송확인 vs 구매진행\n"
        "아직 구매 전이라도 '언제 배송되는가'가 질문의 핵심이면 배송확인입니다.\n"
        "상품을 실제로 결제하거나 구매 완료하기 위한 문제가 핵심이면 구매진행입니다.\n\n"


        "⑤ 구매진행 vs 서비스이용\n"
        "결제를 완료하기 위해 문제를 해결하는 경우 → 구매진행\n"
        "이미 구매한 상품·강의·서비스를 사용하는 과정의 문제 → 서비스이용\n\n"


        "[출력 규칙]\n"
        "반드시 다음 7개 중 하나만 출력하세요.\n"
        "환불요청 / 주문취소 / 불만제기 / 배송확인 / 교환반품 / 구매진행 / 서비스이용\n"
        "설명이나 이유를 출력하지 마세요.\n"

        "<|im_end|>\n"

        "<|im_start|>user\n"

        f"{blocks}\n"

        "이 고객 통화의 카테고리를 하나만 선택하세요.\n"

        "<|im_end|>\n"

        # Thinking 출력 방지
        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    )


# ============================================================
# 3. 출력 Parser
# ============================================================

def parse_category(text):

    if text is None:
        return None

    text = str(text).strip()

    # 정확히 카테고리 하나만 나온 경우
    if text in CATEGORIES:
        return text

    # 혹시 "카테고리: 환불요청"처럼 나온 경우 대응
    for category in CATEGORIES:

        if category in text:
            return category

    return None


# ============================================================
# 4. 단일 call 추론
# ============================================================

def classify_call_redefined(call_id):

    utts = get_call_utts(call_id)

    if not utts:

        return {
            "pred": None,
            "raw": "",
            "error": "no_utts",
        }

    try:

        # ----------------------------------------
        # Audio load
        # ----------------------------------------

        audios = []

        for audio_path, _ in utts:

            wav, _ = librosa.load(
                audio_path,
                sr=16000,
                mono=True,
            )

            audios.append(
                (wav, 16000)
            )


        # ----------------------------------------
        # Prompt
        # ----------------------------------------

        prompt = build_prompt_redefined(
            utts
        )


        # ----------------------------------------
        # Generate
        # ----------------------------------------

        out = llm.generate(
            [
                {
                    "prompt": prompt,
                    "multi_modal_data": {
                        "audio": audios
                    }
                }
            ],
            sampling_params=sampling_redefined,
            use_tqdm=False,
        )


        raw = (
            out[0]
            .outputs[0]
            .text
            .strip()
        )


        pred = parse_category(
            raw
        )


        return {
            "pred": pred,
            "raw": raw,
            "error": None,
        }


    except Exception as e:

        return {
            "pred": None,
            "raw": "",
            "error": repr(e),
        }


# ============================================================
# 5. Test set
#
# 기존 df_tx의 call_id / gold 그대로 사용
# ============================================================

test_df = (
    df_tx[
        ["call_id", "gold"]
    ]
    .drop_duplicates("call_id")
    .reset_index(drop=True)
    .copy()
)


print("=" * 70)
print("재실험 시작")
print("=" * 70)

print(
    f"Test samples: "
    f"{len(test_df)}"
)


# ============================================================
# 6. 전체 inference
# ============================================================

results = []

t0 = time.time()


for idx, row in test_df.iterrows():

    cid = row["call_id"]
    gold = row["gold"]


    result = classify_call_redefined(
        cid
    )


    results.append(
        {
            "call_id": cid,
            "gold": gold,
            "pred": result["pred"],
            "raw": result["raw"],
            "error": result["error"],
        }
    )


    n = idx + 1


    if (
        n % 10 == 0
        or n == len(test_df)
    ):

        elapsed = (
            time.time()
            - t0
        )


        valid = sum(
            x["pred"] is not None
            for x in results
        )


        fail = (
            len(results)
            - valid
        )


        print(
            f"{n}/{len(test_df)} "
            f"| {elapsed:.0f}s "
            f"| valid={valid} "
            f"| fail={fail}"
        )


# ============================================================
# 7. 결과 DataFrame
# ============================================================

df_tx_v2 = pd.DataFrame(
    results
)


print("\n완료.")

print(
    f"valid prediction: "
    f"{df_tx_v2['pred'].notna().sum()}"
)

print(
    f"parse/error: "
    f"{df_tx_v2['pred'].isna().sum()}"
)


# ============================================================
# 8. 평가 함수
# ============================================================

def evaluate_df(
    df,
    pred_col="pred",
    title=""
):

    d = (
        df.dropna(
            subset=[
                "gold",
                pred_col
            ]
        )
        .copy()
    )


    macro = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="macro",
        zero_division=0,
    )


    weighted = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="weighted",
        zero_division=0,
    )


    report = classification_report(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        output_dict=True,
        zero_division=0,
    )


    cm = confusion_matrix(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
    )


    cm_df = pd.DataFrame(
        cm,
        index=[
            f"g_{x}"
            for x in CATEGORIES
        ],
        columns=[
            f"p_{x}"
            for x in CATEGORIES
        ],
    )


    print("\n" + "=" * 80)

    print(title)

    print("=" * 80)

    print(
        f"macro-F1    : "
        f"{macro:.3f}"
    )

    print(
        f"weighted-F1 : "
        f"{weighted:.3f}"
    )


    print("\n클래스별 성능")

    for c in CATEGORIES:

        print(
            f"{c:<8} "
            f"P={report[c]['precision']:.3f} "
            f"R={report[c]['recall']:.3f} "
            f"F1={report[c]['f1-score']:.3f} "
            f"N={int(report[c]['support'])}"
        )


    print(
        "\nConfusion Matrix"
    )

    print(
        cm_df
    )


    return {
        "macro_f1": macro,
        "weighted_f1": weighted,
        "report": report,
        "cm": cm,
        "cm_df": cm_df,
    }


# ============================================================
# 9. 기존 Neutral vs 새로운 정의 비교
# ============================================================

res_old = evaluate_df(
    df_tx,
    pred_col="pred",
    title="① 기존 Neutral (df_tx)"
)


res_new = evaluate_df(
    df_tx_v2,
    pred_col="pred",
    title="② 개선된 카테고리 정의 (df_tx_v2)"
)


# ============================================================
# 10. 클래스별 F1 전후 비교
# ============================================================

comparison_rows = []


for c in CATEGORIES:

    old_f1 = (
        res_old["report"][c]["f1-score"]
    )

    new_f1 = (
        res_new["report"][c]["f1-score"]
    )

    old_recall = (
        res_old["report"][c]["recall"]
    )

    new_recall = (
        res_new["report"][c]["recall"]
    )


    comparison_rows.append(
        {
            "category": c,
            "old_f1": old_f1,
            "new_f1": new_f1,
            "delta_f1": new_f1 - old_f1,
            "old_recall": old_recall,
            "new_recall": new_recall,
            "delta_recall": new_recall - old_recall,
        }
    )


df_compare = pd.DataFrame(
    comparison_rows
)


print(
    "\n"
    + "=" * 80
)

print(
    "클래스별 전후 변화"
)

print(
    "=" * 80
)


print(
    df_compare
    .round(3)
    .to_string(index=False)
)


print(
    "\nMacro-F1 변화:"
)

print(
    f"{res_old['macro_f1']:.3f} "
    f"→ "
    f"{res_new['macro_f1']:.3f} "
    f"("
    f"{res_new['macro_f1'] - res_old['macro_f1']:+.3f}"
    f")"
)


# ============================================================
# 11. 주문취소만 상세 분석
# ============================================================

cancel_cases = (
    df_tx_v2[
        df_tx_v2["gold"]
        == "주문취소"
    ]
    .copy()
)


# 기존 prediction 붙이기
old_pred_map = (
    df_tx[
        ["call_id", "pred"]
    ]
    .rename(
        columns={
            "pred": "pred_old"
        }
    )
)


cancel_cases = cancel_cases.merge(
    old_pred_map,
    on="call_id",
    how="left",
)


print(
    "\n"
    + "=" * 80
)

print(
    "Gold=주문취소 상세"
)

print(
    "=" * 80
)


print(
    cancel_cases[
        [
            "call_id",
            "gold",
            "pred_old",
            "pred"
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\n기존 주문취소 예측 분포:"
)

print(
    cancel_cases[
        "pred_old"
    ]
    .value_counts()
)


print(
    "\n새 주문취소 예측 분포:"
)

print(
    cancel_cases[
        "pred"
    ]
    .value_counts()
)


# ============================================================
# 12. prediction이 바뀐 샘플 분석
# ============================================================

all_compare = (
    df_tx_v2[
        [
            "call_id",
            "gold",
            "pred",
            "raw"
        ]
    ]
    .rename(
        columns={
            "pred": "pred_new"
        }
    )
    .merge(
        df_tx[
            [
                "call_id",
                "pred"
            ]
        ]
        .rename(
            columns={
                "pred": "pred_old"
            }
        ),
        on="call_id",
        how="left",
    )
)


all_compare[
    "old_correct"
] = (
    all_compare[
        "pred_old"
    ]
    ==
    all_compare[
        "gold"
    ]
)


all_compare[
    "new_correct"
] = (
    all_compare[
        "pred_new"
    ]
    ==
    all_compare[
        "gold"
    ]
)


all_compare[
    "changed"
] = (
    all_compare[
        "pred_old"
    ]
    !=
    all_compare[
        "pred_new"
    ]
)


changed = (
    all_compare[
        all_compare[
            "changed"
        ]
    ]
    .copy()
)


# 변화 유형
def classify_change(row):

    if (
        not row["old_correct"]
        and row["new_correct"]
    ):
        return "FIXED"

    if (
        row["old_correct"]
        and not row["new_correct"]
    ):
        return "BROKEN"

    if (
        not row["old_correct"]
        and not row["new_correct"]
    ):
        return "CHANGED_WRONG"

    return "SAME_CORRECT"


changed[
    "change_type"
] = changed.apply(
    classify_change,
    axis=1
)


print(
    "\n"
    + "=" * 80
)

print(
    f"Prediction 변경: "
    f"{len(changed)}건"
)

print(
    "=" * 80
)


print(
    changed[
        "change_type"
    ]
    .value_counts()
)


# ============================================================
# 13. 무엇을 고쳤고 무엇을 망쳤는지 확인
# ============================================================

print(
    "\n[FIXED: 기존 오답 → 새 정답]"
)

fixed = (
    changed[
        changed[
            "change_type"
        ]
        ==
        "FIXED"
    ]
)


if len(fixed) > 0:

    print(
        fixed[
            [
                "call_id",
                "gold",
                "pred_old",
                "pred_new"
            ]
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "없음"
    )


print(
    "\n[BROKEN: 기존 정답 → 새 오답]"
)

broken = (
    changed[
        changed[
            "change_type"
        ]
        ==
        "BROKEN"
    ]
)


if len(broken) > 0:

    print(
        broken[
            [
                "call_id",
                "gold",
                "pred_old",
                "pred_new"
            ]
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "없음"
    )


# ============================================================
# 14. 정의 변경으로 가장 많이 발생한 이동 확인
# ============================================================

transition = (
    changed
    .groupby(
        [
            "pred_old",
            "pred_new"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "Prediction 이동"
)

print(
    "=" * 80
)


print(
    transition
    .to_string(
        index=False
    )
)


# ============================================================
# 15. 저장용 최종 DataFrame
# ============================================================

df_tx_v2_analysis = (
    all_compare
    .copy()
)


print(
    "\n실험 완료."
)

print(
    "새 결과           : df_tx_v2"
)

print(
    "기존/신규 비교    : df_compare"
)

print(
    "샘플 단위 비교    : df_tx_v2_analysis"
)

[INFO] 기존 sampling_tx 그대로 사용
재실험 시작
Test samples: 250
10/250 | 6s | valid=10 | fail=0
20/250 | 11s | valid=20 | fail=0
30/250 | 16s | valid=30 | fail=0
40/250 | 21s | valid=40 | fail=0
50/250 | 27s | valid=50 | fail=0
60/250 | 33s | valid=60 | fail=0
70/250 | 39s | valid=70 | fail=0
80/250 | 45s | valid=80 | fail=0
90/250 | 51s | valid=90 | fail=0
100/250 | 57s | valid=100 | fail=0
110/250 | 63s | valid=110 | fail=0
120/250 | 69s | valid=120 | fail=0
130/250 | 75s | valid=130 | fail=0
140/250 | 81s | valid=140 | fail=0
150/250 | 87s | valid=150 | fail=0
160/250 | 92s | valid=160 | fail=0
170/250 | 98s | valid=170 | fail=0
180/250 | 104s | valid=180 | fail=0
190/250 | 110s | valid=190 | fail=0
200/250 | 116s | valid=200 | fail=0
210/250 | 122s | valid=210 | fail=0
220/250 | 127s | valid=220 | fail=0
230/250 | 133s | valid=230 | fail=0
240/250 | 139s | valid=240 | fail=0
250/250 | 145s | valid=250 | fail=0

완료.
valid prediction: 250
parse/error: 0

① 기존 Neutral (df_tx)
macro-F1    : 0.48

In [27]:
# ============================================================
# FINAL ZERO-SHOT EXPERIMENT
#
# Base     : df_tx_v2  (개선된 category definition)
# Detector : df_strong (strong complaint prompt)
#
# Strong=불만 & Base!=불만인 경우만
# complaint score 0~4 다시 추론
#
# → threshold 이상이면 불만 override
# → 아니면 df_tx_v2 유지
# ============================================================

import time
import re
import librosa
import pandas as pd
import numpy as np

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

from vllm import SamplingParams


# ============================================================
# 0. 설정
# ============================================================

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]

COMPLAINT = "불만제기"


assert "df_tx_v2" in globals(), "df_tx_v2가 없습니다."
assert "df_strong" in globals(), "df_strong이 없습니다."
assert "llm" in globals(), "llm이 없습니다."
assert "get_call_utts" in globals(), "get_call_utts()가 없습니다."


# ============================================================
# 1. Sampling
# ============================================================

sampling_score_v2 = SamplingParams(
    temperature=0.0,
    max_tokens=4,
    stop=["\n", "<|im_end|>"],
)


# ============================================================
# 2. 클래스별 비교 가이드
# ============================================================

BASE_GUIDANCE = {

    "환불요청":
        "환불요청은 이미 진행된 거래에서 금전 반환이 핵심 목적입니다. "
        "단순히 환불을 강하게 요구하거나 짜증스럽게 말한다는 이유만으로 "
        "불만제기로 판단하지 않습니다.",

    "주문취소":
        "주문취소는 상품이나 서비스가 본격적으로 이행되기 전에 "
        "기존 주문 자체를 없애는 것이 핵심 목적입니다. "
        "잘못 주문, 재주문, 합배송, 배송비 절약 목적의 취소 등이 포함됩니다.",

    "배송확인":
        "배송확인은 상품의 발송 여부, 위치, 도착 시점, 배송 소요일 등을 "
        "확인하는 것이 핵심입니다. 배송이 늦어 답답해하는 것만으로 "
        "불만제기가 되지는 않습니다.",

    "교환반품":
        "교환반품은 파본, 오배송, 누락, 훼손 등 받은 상품 자체의 문제로 "
        "물건을 교환하거나 반품하는 것이 핵심입니다.",

    "구매진행":
        "구매진행은 상품이나 강의의 구매·결제를 완료하는 것이 핵심입니다. "
        "결제 오류나 보안 프로그램 문제라도 구매 완료가 목적이면 구매진행입니다.",

    "서비스이용":
        "서비스이용은 이미 구매·등록된 서비스나 콘텐츠를 사용하는 과정의 "
        "문제 해결이나 이용 문의가 핵심입니다.",
}


# ============================================================
# 3. Complaint score prompt
# ============================================================

def build_prompt_score_v2(utts, base_pred):

    blocks = ""

    for i, (_, txt) in enumerate(utts, 1):

        txt = "" if txt is None else str(txt)

        blocks += (
            f"발화{i}: "
            f"<|audio_start|><|audio_pad|><|audio_end|> "
            f'전사: "{txt}"\n'
        )

    guidance = BASE_GUIDANCE.get(
        base_pred,
        f"{base_pred}와 불만제기의 차이를 고객의 실제 목적을 중심으로 판단하세요."
    )

    return (
        "<|im_start|>system\n"

        "당신은 한국어 콜센터 고객 발화에서 "
        "불만·항의가 실제로 존재하는 정도를 평가합니다.\n"
        "음성과 전사를 함께 사용하세요.\n\n"

        "중요: 최종 카테고리를 직접 분류하지 말고 "
        "불만·항의 증거의 강도만 평가합니다.\n\n"


        "[불만 강도 점수]\n"

        "0 = 불만·항의가 없음. 일반적인 요청 또는 문의.\n"

        "1 = 약한 불편함이나 짜증은 있지만 "
        "독립적인 항의나 문제 제기로 보기 어려움.\n"

        "2 = 불만 가능성이 있으나 애매함. "
        "부정적인 어조 또는 문제 언급은 존재하지만 "
        "단순 요청·문의와 명확히 구분하기 어려움.\n"

        "3 = 명확한 불만·항의. "
        "고객이 처리 지연, 반복 문의, 약속 불이행, 연락 두절, "
        "상품·서비스·상담 문제 등을 실제로 따지거나 비판함.\n"

        "4 = 매우 강한 불만·항의. "
        "반복적인 항의, 심한 격앙, 책임 추궁, 강한 비판, "
        "즉각적인 해결 요구 등이 명백함.\n\n"


        "[중요한 판단 원칙]\n"

        "- 목소리가 크거나 짜증스럽다는 이유만으로 높은 점수를 주지 마세요.\n"

        "- 환불, 주문취소, 배송확인, 교환반품 등을 강한 말투로 요구하는 것만으로 "
        "불만제기가 되는 것은 아닙니다.\n"

        "- 처리 지연, 반복 문의, 연락 미응답, 약속 불이행 등 "
        "'기존 문제에 대한 항의'가 독립적으로 존재하는지를 중요하게 보세요.\n"

        "- 반대로 목소리가 차분하더라도 문제를 반복적으로 따지고 "
        "서비스나 상담 대응을 비판한다면 높은 점수를 줄 수 있습니다.\n"

        "- 음성 어조는 중요한 보조 신호이지만 "
        "반드시 전사 내용과 함께 판단하세요.\n\n"


        "[현재 semantic classifier 결과]\n"

        f"{base_pred}\n"

        f"판단 참고: {guidance}\n\n"


        "[출력 규칙]\n"

        "반드시 0, 1, 2, 3, 4 중 숫자 하나만 출력하세요.\n"

        "<|im_end|>\n"

        "<|im_start|>user\n"

        f"{blocks}\n"

        f"기본 의미 분류 결과는 '{base_pred}'입니다.\n"

        "이 통화에서 고객의 불만·항의 증거 강도를 "
        "0~4 중 하나로 평가하세요.\n"

        "<|im_end|>\n"

        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    )


# ============================================================
# 4. parser
# ============================================================

def parse_score_v2(text):

    if text is None:
        return None

    text = str(text).strip()

    m = re.search(r"\b([0-4])\b", text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# 5. 단일 call scoring
# ============================================================

def score_complaint_v2(call_id, base_pred):

    utts = get_call_utts(call_id)

    if not utts:

        return {
            "score": None,
            "raw": "",
            "error": "no_utts"
        }

    try:

        audios = []

        for audio_path, _ in utts:

            wav, _ = librosa.load(
                audio_path,
                sr=16000,
                mono=True
            )

            audios.append(
                (wav, 16000)
            )


        prompt = build_prompt_score_v2(
            utts,
            base_pred
        )


        out = llm.generate(
            [
                {
                    "prompt": prompt,
                    "multi_modal_data": {
                        "audio": audios
                    }
                }
            ],
            sampling_params=sampling_score_v2,
            use_tqdm=False,
        )


        raw = (
            out[0]
            .outputs[0]
            .text
            .strip()
        )


        score = parse_score_v2(raw)


        return {
            "score": score,
            "raw": raw,
            "error": None
        }


    except Exception as e:

        return {
            "score": None,
            "raw": "",
            "error": repr(e)
        }


# ============================================================
# 6. df_tx_v2 + df_strong merge
# ============================================================

base = (
    df_tx_v2[
        ["call_id", "gold", "pred"]
    ]
    .copy()
    .rename(
        columns={
            "pred": "pred_base"
        }
    )
)


strong = (
    df_strong[
        ["call_id", "gold", "pred"]
    ]
    .copy()
    .rename(
        columns={
            "gold": "gold_strong",
            "pred": "pred_strong"
        }
    )
)


df_final_exp = base.merge(
    strong,
    on="call_id",
    how="left"
)


print(
    f"merge shape: "
    f"{df_final_exp.shape}"
)


# gold 체크
gold_mismatch = (
    df_final_exp["gold_strong"].notna()
    &
    (
        df_final_exp["gold"]
        !=
        df_final_exp["gold_strong"]
    )
)


print(
    f"gold mismatch: "
    f"{gold_mismatch.sum()}"
)


# ============================================================
# 7. 새 disagreement 정의
#
# Strong = 불만
# Base(df_tx_v2) != 불만
#
# 이 경우만 score 추론
# ============================================================

df_final_exp["need_score"] = (
    (df_final_exp["pred_strong"] == COMPLAINT)
    &
    (df_final_exp["pred_base"] != COMPLAINT)
    &
    (df_final_exp["pred_base"].notna())
)


target = (
    df_final_exp[
        df_final_exp["need_score"]
    ]
    .copy()
)


print("\n" + "=" * 70)

print("새 Complaint scoring 대상")

print("=" * 70)


print(
    f"Strong 불만 예측: "
    f"{(df_final_exp['pred_strong'] == COMPLAINT).sum()}"
)


print(
    f"Base도 불만: "
    f"{(
        (df_final_exp['pred_strong'] == COMPLAINT)
        &
        (df_final_exp['pred_base'] == COMPLAINT)
    ).sum()}"
)


print(
    f"새 Score 대상: "
    f"{len(target)}"
)


print("\nBase prediction 분포")

print(
    target["pred_base"]
    .value_counts()
)


# ============================================================
# 8. Complaint score 재추론
# ============================================================

score_results_v2 = {}


print("\nScoring 시작...")


t0 = time.time()


for idx, (_, row) in enumerate(
    target.iterrows(),
    1
):

    cid = row["call_id"]

    base_pred = row["pred_base"]


    result = score_complaint_v2(
        call_id=cid,
        base_pred=base_pred
    )


    score_results_v2[cid] = result


    if (
        idx % 10 == 0
        or
        idx == len(target)
    ):

        valid = sum(
            x["score"] is not None
            for x in score_results_v2.values()
        )

        fail = (
            idx - valid
        )

        print(
            f"{idx}/{len(target)} "
            f"| {time.time()-t0:.0f}s "
            f"| valid={valid} "
            f"| fail={fail}"
        )


# ============================================================
# 9. score 저장
# ============================================================

df_final_exp["complaint_score"] = np.nan

df_final_exp["score_raw"] = None

df_final_exp["score_error"] = None


for i, row in df_final_exp.iterrows():

    cid = row["call_id"]

    if cid not in score_results_v2:
        continue

    result = score_results_v2[cid]

    df_final_exp.at[
        i,
        "complaint_score"
    ] = result["score"]

    df_final_exp.at[
        i,
        "score_raw"
    ] = result["raw"]

    df_final_exp.at[
        i,
        "score_error"
    ] = result["error"]


print("\nScore 분포")

print(
    df_final_exp.loc[
        df_final_exp["need_score"],
        "complaint_score"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 10. Base-first fusion
#
# 기본 = df_tx_v2
#
# Strong=불만이며 score >= threshold인 경우에만
# 불만으로 override
# ============================================================

def make_final_pred_v2(row, threshold):

    base_pred = row["pred_base"]

    strong_pred = row["pred_strong"]

    score = row["complaint_score"]


    # 기본은 semantic base
    if pd.isna(base_pred):
        return strong_pred


    # Base 자체가 불만이면 유지
    if base_pred == COMPLAINT:
        return COMPLAINT


    # Strong이 불만이라고 탐지했을 때만 override 검토
    if strong_pred == COMPLAINT:

        if pd.isna(score):
            return base_pred

        if score >= threshold:
            return COMPLAINT


    return base_pred


# ============================================================
# 11. Threshold sweep
# ============================================================

sweep_results_v2 = []


for threshold in [0, 1, 2, 3, 4, 5]:

    pred_col = f"pred_t{threshold}"

    df_final_exp[pred_col] = (
        df_final_exp.apply(
            lambda row: make_final_pred_v2(
                row,
                threshold
            ),
            axis=1
        )
    )


    d = (
        df_final_exp.dropna(
            subset=[
                "gold",
                pred_col
            ]
        )
    )


    macro = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    )


    weighted = f1_score(
        d["gold"],
        d[pred_col],
        labels=CATEGORIES,
        average="weighted",
        zero_division=0
    )


    complaint_precision = precision_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0
    )


    complaint_recall = recall_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0
    )


    complaint_f1 = f1_score(
        d["gold"],
        d[pred_col],
        labels=[COMPLAINT],
        average="micro",
        zero_division=0
    )


    tp = (
        (d["gold"] == COMPLAINT)
        &
        (d[pred_col] == COMPLAINT)
    ).sum()


    fp = (
        (d["gold"] != COMPLAINT)
        &
        (d[pred_col] == COMPLAINT)
    ).sum()


    fn = (
        (d["gold"] == COMPLAINT)
        &
        (d[pred_col] != COMPLAINT)
    ).sum()


    sweep_results_v2.append(
        {
            "threshold": threshold,
            "macro_f1": macro,
            "weighted_f1": weighted,
            "complaint_precision": complaint_precision,
            "complaint_recall": complaint_recall,
            "complaint_f1": complaint_f1,
            "complaint_TP": tp,
            "complaint_FP": fp,
            "complaint_FN": fn,
        }
    )


df_sweep_v2strong = pd.DataFrame(
    sweep_results_v2
)


print(
    "\n"
    + "=" * 100
)

print(
    "df_tx_v2 + Strong + Complaint Score Threshold Sweep"
)

print(
    "=" * 100
)


print(
    df_sweep_v2strong
    .round(3)
    .to_string(index=False)
)


# ============================================================
# 12. Best threshold
# ============================================================

best_macro = (
    df_sweep_v2strong.loc[
        df_sweep_v2strong[
            "macro_f1"
        ].idxmax()
    ]
)


best_complaint = (
    df_sweep_v2strong.loc[
        df_sweep_v2strong[
            "complaint_f1"
        ].idxmax()
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "Best Threshold"
)

print(
    "=" * 100
)


print(
    f"Macro-F1 기준 threshold = "
    f"{int(best_macro['threshold'])}"
)

print(
    f"macro-F1      = "
    f"{best_macro['macro_f1']:.3f}"
)

print(
    f"weighted-F1   = "
    f"{best_macro['weighted_f1']:.3f}"
)

print(
    f"complaint F1  = "
    f"{best_macro['complaint_f1']:.3f}"
)

print(
    f"complaint R   = "
    f"{best_macro['complaint_recall']:.3f}"
)

print(
    f"complaint P   = "
    f"{best_macro['complaint_precision']:.3f}"
)


print()


print(
    f"Complaint-F1 기준 threshold = "
    f"{int(best_complaint['threshold'])}"
)

print(
    f"macro-F1      = "
    f"{best_complaint['macro_f1']:.3f}"
)

print(
    f"complaint F1  = "
    f"{best_complaint['complaint_f1']:.3f}"
)

print(
    f"complaint R   = "
    f"{best_complaint['complaint_recall']:.3f}"
)


# ============================================================
# 13. Macro best 선택
# ============================================================

BEST_THRESHOLD_V2 = int(
    best_macro["threshold"]
)


df_final_exp["pred_final"] = (
    df_final_exp[
        f"pred_t{BEST_THRESHOLD_V2}"
    ]
)


# ============================================================
# 14. Confusion matrix
# ============================================================

cm = confusion_matrix(
    df_final_exp["gold"],
    df_final_exp["pred_final"],
    labels=CATEGORIES
)


cm_df = pd.DataFrame(
    cm,
    index=[
        f"g_{x}"
        for x in CATEGORIES
    ],
    columns=[
        f"p_{x}"
        for x in CATEGORIES
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    f"최종 Confusion Matrix "
    f"(threshold={BEST_THRESHOLD_V2})"
)

print(
    "=" * 100
)


print(
    cm_df
)


# ============================================================
# 15. 클래스별 성능
# ============================================================

report = classification_report(
    df_final_exp["gold"],
    df_final_exp["pred_final"],
    labels=CATEGORIES,
    output_dict=True,
    zero_division=0
)


print("\n클래스별 성능")


for c in CATEGORIES:

    print(
        f"{c:<8} "
        f"P={report[c]['precision']:.3f} "
        f"R={report[c]['recall']:.3f} "
        f"F1={report[c]['f1-score']:.3f}"
    )


# ============================================================
# 16. 지금까지 실험과 비교
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "Zero-shot 실험 비교"
)

print(
    "=" * 100
)


summary = pd.DataFrame(
    [
        {
            "model": "Original Neutral",
            "macro_f1": 0.482,
            "complaint_f1": 0.444,
            "complaint_recall": 0.400,
        },
        {
            "model": "Strong",
            "macro_f1": 0.421,
            "complaint_f1": 0.563,
            "complaint_recall": 0.760,
        },
        {
            "model": "Original Neutral + Strong fusion",
            "macro_f1": 0.490,
            "complaint_f1": 0.544,
            "complaint_recall": 0.620,
        },
        {
            "model": "Redefined Neutral",
            "macro_f1": 0.453,
            "complaint_f1": 0.233,
            "complaint_recall": 0.140,
        },
        {
            "model": "Redefined + Strong fusion",
            "macro_f1": best_macro["macro_f1"],
            "complaint_f1": best_macro["complaint_f1"],
            "complaint_recall": best_macro["complaint_recall"],
        },
        {
            "model": "GPT baseline",
            "macro_f1": 0.525,
            "complaint_f1": 0.281,
            "complaint_recall": 0.180,
        },
    ]
)


print(
    summary
    .round(3)
    .to_string(index=False)
)


# ============================================================
# 17. score가 실제 불만과 얼마나 대응하는지
# ============================================================

scored = (
    df_final_exp[
        df_final_exp["need_score"]
    ]
    .copy()
)


scored["is_gold_complaint"] = (
    scored["gold"]
    == COMPLAINT
)


print(
    "\n"
    + "=" * 100
)

print(
    "Complaint Score × 실제 Gold"
)

print(
    "=" * 100
)


print(
    pd.crosstab(
        scored["complaint_score"],
        scored["is_gold_complaint"],
        margins=True
    )
)


score_quality = (
    scored
    .dropna(
        subset=["complaint_score"]
    )
    .groupby(
        "complaint_score"
    )
    .agg(
        n=(
            "call_id",
            "size"
        ),
        gold_complaint=(
            "is_gold_complaint",
            "sum"
        ),
        complaint_rate=(
            "is_gold_complaint",
            "mean"
        )
    )
    .reset_index()
)


print(
    "\nScore별 실제 불만 비율"
)


print(
    score_quality
    .round(3)
    .to_string(index=False)
)


# ============================================================
# 18. 최종 분석용 DataFrame
# ============================================================

analysis_cols = [
    "call_id",
    "gold",
    "pred_base",
    "pred_strong",
    "need_score",
    "complaint_score",
    "pred_final",
    "score_raw",
    "score_error",
]


df_final_zero_shot = (
    df_final_exp[
        analysis_cols
    ]
    .copy()
)


print("\n완료.")

print(
    "threshold 결과 : df_sweep_v2strong"
)

print(
    "최종 결과      : df_final_zero_shot"
)

merge shape: (250, 5)
gold mismatch: 0

새 Complaint scoring 대상
Strong 불만 예측: 85
Base도 불만: 9
새 Score 대상: 76

Base prediction 분포
pred_base
환불요청     22
배송확인     20
교환반품     17
구매진행      7
서비스이용     6
주문취소      4
Name: count, dtype: int64

Scoring 시작...
10/76 | 3s | valid=10 | fail=0
20/76 | 6s | valid=20 | fail=0
30/76 | 10s | valid=30 | fail=0
40/76 | 13s | valid=40 | fail=0
50/76 | 16s | valid=50 | fail=0
60/76 | 19s | valid=60 | fail=0
70/76 | 22s | valid=70 | fail=0
76/76 | 24s | valid=76 | fail=0

Score 분포
complaint_score
0.0     2
2.0    49
3.0    25
Name: count, dtype: int64

df_tx_v2 + Strong + Complaint Score Threshold Sweep
 threshold  macro_f1  weighted_f1  complaint_precision  complaint_recall  complaint_f1  complaint_TP  complaint_FP  complaint_FN
         0     0.477        0.557                0.442              0.76         0.559            38            48            12
         1     0.474        0.552                0.440              0.74         0.552            37   